# Proyek Akhir Praktikum Kecerdasan Buatan
## Klasifikasi Jenis Kulit Wajah untuk Rekomendasi Skincare — Custom CNN

**Tipe tugas:** Supervised learning, klasifikasi citra (kategorikal)
**Model:** Custom CNN dari nol (tanpa transfer learning)
**Deployment:** Web app SkinSense (Flask + HTML) di Hugging Face Spaces

### Cara menjalankan
1. Buka di **Google Colab** → `Runtime > Change runtime type > T4 GPU`.
2. Identitas di sel **Konfigurasi** sudah terisi; ubah bila perlu.
3. `Runtime > Run all`. Total waktu ±20–30 menit di GPU T4.
   Di bagian akhir (bagian 9) notebook meminta **token Hugging Face** untuk deploy web (Enter untuk melewati).
4. Di akhir, notebook otomatis membuat:
   - `PA_outputs/Laporan_PA_KB.pdf` → laporan otomatis berisi semua hasil
   - `PA_outputs/model/skin_type_cnn.keras` + `labels.json` → dipakai web app
   - Link demo publik Hugging Face Spaces (bagian 9)
   - `PA_outputs/figures/*.png` → semua grafik
   - `PA_outputs.zip` → semua hasil (terunduh otomatis di Colab)

### Alur notebook
| No | Tahap | Bobot rubrik |
|---|---|---|
| 1 | Pengumpulan data (download Kaggle, cek isi dataset) | 10% |
| 2 | Pemeriksaan kualitas & pembersihan data (citra rusak, duplikat, near-duplicate) | 20% |
| 3 | Analisis & visualisasi dataset (EDA) | 5% |
| 4 | Prapemrosesan: split stratified-group, resize, augmentasi, class weight | 20% |
| 5 | Arsitektur custom CNN, hiperparameter, training | 25% |
| 6 | Evaluasi testing, uji ketangguhan (robustness), Grad-CAM | 25% |
| 7 | (Bonus) Optimisasi hiperparameter (random search) | bonus |
| 8 | Ekspor model untuk deployment + laporan progress PDF | — |

## 0. Setup library

In [ ]:
import os, sys, json, time, random, hashlib, shutil, textwrap, warnings, zipfile, subprocess, io
from pathlib import Path
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image, ImageOps

warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
try:
    import kagglehub
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
    import kagglehub

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 100
print("Python", sys.version.split()[0], "| Colab:", IN_COLAB)

## Konfigurasi
Semua hiperparameter dan pengaturan dikumpulkan di satu tempat supaya mudah dijelaskan dan diubah.

In [ ]:
# ===== Identitas (isi sebelum Run all) =====
JUDUL     = "Klasifikasi Jenis Kulit Wajah untuk Rekomendasi Skincare"
KELOMPOK  = "Henry Moky (2409106070)"
ANGGOTA   = [  # (nama, NIM, tugas utama)
    ("Henry Moky", "2409106070", "Pengumpulan & pembersihan data, EDA, prapemrosesan, model CNN, evaluasi, deployment"),
]
ASLAB     = "..."

# ===== Dataset =====
PRIMARY_DATASET   = "shakyadissanayake/oily-dry-and-normal-skin-types-dataset"  # Oily, Dry and Normal Skin Types Dataset
FALLBACK_DATASET  = None   # tidak dipakai
LOCAL_DATASET_DIR = None   # isi path folder jika dataset di-upload manual (mis. "/content/drive/MyDrive/skin")
MIN_IMAGES_TOTAL  = 100    # pengaman: dataset harus berisi minimal sekian citra

# ===== Pembersihan data =====
MIN_SIDE          = 64     # citra dengan sisi < 64 px dibuang
NEAR_DUP_HAMMING  = 12     # ambang jarak Hamming dHash 16x16 (0-256 bit) untuk kandidat near-duplicate
NEAR_DUP_PIXEL    = 8.0    # konfirmasi: rata-rata selisih piksel grayscale 32x32 (0-255) harus < nilai ini
MIN_PER_CLASS     = 20     # kelas dengan citra < ini dibuang

# ===== Prapemrosesan & training =====
SEED          = 42
IMG_SIZE      = 224
BATCH_SIZE    = 32
EPOCHS        = 60
TEST_FRAC     = 1/7        # ~14% test
VAL_FRAC      = 1/6        # ~14% validasi (dari sisa train+val)
LABEL_SMOOTH  = 0.1

# Hiperparameter model default (bisa ditimpa hasil HPO)
HP = dict(filters=(32, 64, 128, 256), dense=256, dropout=0.4, l2=1e-4, lr=1e-3)

# ===== Bonus: optimisasi hiperparameter =====
RUN_HPO     = False        # True = jalankan random search (±20-25 menit tambahan di T4)
HPO_TRIALS  = 6
HPO_EPOCHS  = 15

# ===== Output =====
OUT_DIR   = Path("PA_outputs");  FIG_DIR = OUT_DIR / "figures";  MODEL_DIR = OUT_DIR / "model"
WORK_DIR  = Path("data_work")
for d in (OUT_DIR, FIG_DIR, MODEL_DIR, WORK_DIR):
    d.mkdir(parents=True, exist_ok=True)

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); os.environ["PYTHONHASHSEED"] = str(s)
set_seed()

FIGS = {}      # nama -> path gambar (dipakai laporan PDF)
REPORT = {}    # angka-angka penting untuk laporan
def savefig(name, caption=""):
    p = FIG_DIR / f"{name}.png"
    plt.savefig(p, dpi=130, bbox_inches="tight")
    FIGS[name] = (str(p), caption)
    return p

### Import TensorFlow/Keras & cek GPU
TensorFlow/Keras dipakai untuk membangun dan melatih CNN. Sel ini juga mengecek apakah GPU tersedia (training di GPU jauh lebih cepat daripada CPU) dan mengunci *random seed* supaya hasil bisa direproduksi.

In [ ]:
import tensorflow as tf
import keras
from keras import layers, regularizers

keras.utils.set_random_seed(SEED)
AUTOTUNE = tf.data.AUTOTUNE
gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| GPU:", gpus if gpus else "TIDAK ADA (training akan lambat)")
REPORT["env"] = dict(tf=tf.__version__, keras=keras.__version__, gpu=str(gpus[0].name) if gpus else "CPU")

## 1. Pengumpulan Data
**Dataset:** [Oily, Dry and Normal Skin Types Dataset](https://www.kaggle.com/datasets/shakyadissanayake/oily-dry-and-normal-skin-types-dataset)
— citra wajah dengan 3 kelas jenis kulit (oily, dry, normal), berasal dari Roboflow, ±126 MB.
Diunduh langsung dari Kaggle dengan `kagglehub` (dataset publik). Zip di dalam dataset diekstrak otomatis,
lalu jumlah file citra dicek sebelum lanjut.

*Catatan:* dataset awal (`sheinsaudi/skin-type-prediction-dataset`) diganti karena berupa data sintetis tabular, bukan citra.

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def list_images(root):
    return [p for p in Path(root).rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXT]

def extract_zips(root, dest):
    # Ekstrak zip bersarang (beberapa dataset Kaggle menyimpan zip di dalam zip)
    for z in Path(root).rglob("*.zip"):
        target = Path(dest) / z.stem
        if not target.exists():
            print("  ekstrak", z.name, "->", target)
            with zipfile.ZipFile(z) as zf:
                zf.extractall(target)

def download(slug):
    try:
        p = Path(kagglehub.dataset_download(slug))
        print(f"✓ {slug} -> {p}")
        return p
    except Exception as e:
        print(f"✗ gagal download {slug}: {e}")
        return None

def tree(root, max_depth=3, max_items=6):
    root = Path(root)
    for dirpath, dirnames, filenames in os.walk(root):
        depth = Path(dirpath).relative_to(root).parts
        if len(depth) >= max_depth:
            dirnames[:] = []; continue
        ind = "    " * len(depth)
        print(f"{ind}{Path(dirpath).name}/  ({len(filenames)} file)")
        dirnames.sort()
        if len(dirnames) > max_items:
            dirnames[:] = dirnames[:max_items]

def resolve_dataset():
    if LOCAL_DATASET_DIR:
        return [Path(LOCAL_DATASET_DIR)], "local:" + str(LOCAL_DATASET_DIR), {}
    info = {}
    for slug in [x for x in (PRIMARY_DATASET, FALLBACK_DATASET) if x]:
        root = download(slug)
        if root is None:
            info[slug] = "gagal download"; continue
        extract_zips(root, WORK_DIR / slug.replace("/", "__"))
        roots = [root, WORK_DIR / slug.replace("/", "__")]
        n = sum(len(list_images(r)) for r in roots if Path(r).exists())
        others = sorted({p.suffix.lower() for p in Path(root).rglob("*") if p.is_file()} - IMG_EXT)
        info[slug] = f"{n} citra; tipe file lain: {others}"
        print(f"  {slug}: {n} file citra, tipe file lain: {others}")
        if n >= MIN_IMAGES_TOTAL:
            return [r for r in roots if Path(r).exists()], slug, info
        print(f"  ⚠ {slug} bukan dataset citra yang cukup (n={n}). Mencoba dataset berikutnya...")
    raise RuntimeError("Tidak ada dataset citra yang bisa dipakai. Isi LOCAL_DATASET_DIR secara manual.")

DATA_ROOTS, DATASET_USED, DATASET_CHECK = resolve_dataset()
REPORT["dataset_used"] = DATASET_USED
REPORT["dataset_check"] = DATASET_CHECK
print("\nDataset dipakai:", DATASET_USED)
for r in DATA_ROOTS:
    tree(r)

### 1.1 Membaca label dari struktur folder
Label diambil dari nama folder induk (`.../train/oily/img.jpg` → `oily`). Jika dataset berformat Roboflow-CSV
(`_classes.csv` one-hot), label dibaca dari CSV. Split asli (train/valid/test) dicatat, tetapi **tidak dipakai**:
data digabung lalu di-split ulang setelah deduplikasi agar tidak ada kebocoran data (*data leakage*) antar split.

In [ ]:
SPLIT_NAMES = {"train", "training", "valid", "val", "validation", "test", "testing", "images", "data", "dataset"}

def norm_label(s):
    s = str(s).strip().lower().replace("_", " ").replace("-", " ")
    s = " ".join(w for w in s.split() if w not in {"skin", "type", "types"}) or s
    return s

def read_roboflow_csv(root):
    lab = {}
    for csv in Path(root).rglob("_classes.csv"):
        d = pd.read_csv(csv); d.columns = [c.strip() for c in d.columns]
        fcol, ccols = d.columns[0], list(d.columns[1:])
        for _, r in d.iterrows():
            v = r[ccols].astype(float).values
            if v.sum() == 1:
                lab[str((csv.parent / str(r[fcol])).resolve())] = ccols[int(v.argmax())]
    return lab

def scan_dataset(roots):
    rows, seen = [], set()
    for root in roots:
        root = Path(root)
        csv_lab = read_roboflow_csv(root)
        for p in list_images(root):
            rp = str(p.resolve())
            if rp in seen: continue
            seen.add(rp)
            parts = p.parent.relative_to(root).parts
            label = csv_lab.get(rp)
            if label is None:
                cand = [x for x in reversed(parts) if x.lower() not in SPLIT_NAMES]
                label = cand[0] if cand else None
            split = next((x.lower() for x in reversed(parts) if x.lower() in SPLIT_NAMES - {"images", "data", "dataset"}), "-")
            rows.append(dict(path=rp, label=None if label is None else norm_label(label), orig_split=split))
    d = pd.DataFrame(rows)
    return d[d.label.notna()].reset_index(drop=True)

df_raw = scan_dataset(DATA_ROOTS)
print("Total citra terbaca:", len(df_raw))
display(pd.crosstab(df_raw.label, df_raw.orig_split, margins=True))
REPORT["n_raw"] = len(df_raw)
REPORT["raw_per_class"] = df_raw.label.value_counts().to_dict()

## 2. Pemeriksaan Kualitas & Pembersihan Data
Setiap citra diperiksa:
- **Rusak/tidak bisa dibuka** (`PIL.verify`) → dibuang
- **Resolusi terlalu kecil** (< `MIN_SIDE` px) → dibuang
- **Duplikat persis** (hash MD5 sama) → disisakan satu; jika labelnya berbeda (label konflik) → semua dibuang
- **Near-duplicate** (perceptual *difference hash* / dHash, jarak Hamming ≤ `NEAR_DUP_HAMMING`) → dikelompokkan.
  Kelompok dengan label konflik dibuang; kelompok yang konsisten **dijaga berada di split yang sama** (lihat tahap 4).
  Ini penting karena dataset Roboflow sering berisi foto yang sama/serupa di train dan test → akurasi palsu.

Sekaligus dihitung statistik untuk EDA: ukuran, kecerahan, kontras, rata-rata kanal RGB, ketajaman (variance of Laplacian).

In [ ]:
def dhash_hex(img, size=16):
    g = np.asarray(img.convert("L").resize((size + 1, size), Image.BILINEAR), dtype=np.int16)
    return np.packbits((g[:, 1:] > g[:, :-1]).flatten()).tobytes().hex()

def inspect_image(path):
    try:
        raw = open(path, "rb").read()
        md5 = hashlib.md5(raw).hexdigest()
        with Image.open(io.BytesIO(raw)) as im:
            im.verify()
        with Image.open(io.BytesIO(raw)) as im:
            fmt = im.format
            im = ImageOps.exif_transpose(im)
            w, h = im.size; mode = im.mode
            rgb = im.convert("RGB")
        th = np.asarray(rgb.resize((96, 96), Image.BILINEAR), dtype=np.float32)
        gray = cv2.cvtColor(th.astype(np.uint8), cv2.COLOR_RGB2GRAY)
        thumb = np.asarray(rgb.convert("L").resize((32, 32), Image.BILINEAR), dtype=np.uint8).ravel()
        return dict(ok=True, err="", md5=md5, dhash=dhash_hex(rgb), thumb=thumb, width=w, height=h, mode=mode, fmt=fmt,
                    kb=len(raw) / 1024, R=th[..., 0].mean(), G=th[..., 1].mean(), B=th[..., 2].mean(),
                    brightness=th.mean(), contrast=th.std(), sharpness=cv2.Laplacian(gray, cv2.CV_64F).var())
    except Exception as e:
        return dict(ok=False, err=f"{type(e).__name__}: {e}")

t0 = time.time()
with ThreadPoolExecutor(max_workers=8) as ex:
    stats = list(ex.map(inspect_image, df_raw.path))
df_q = pd.concat([df_raw, pd.DataFrame(stats)], axis=1)
print(f"Inspeksi {len(df_q)} citra selesai dalam {time.time() - t0:.1f} s")
display(df_q.drop(columns=["path", "thumb"]).head())

### 2.1 Menerapkan aturan pembersihan
Hasil inspeksi dipakai untuk menyaring data secara berurutan. Setiap langkah dicatat (jumlah citra yang dibuang) dalam tabel log pembersihan.
Near-duplicate dikelompokkan dengan struktur data **Union-Find** sehingga foto-foto yang mirip satu sama lain menjadi satu grup.

In [ ]:
class UnionFind:
    def __init__(self, n): self.p = list(range(n))
    def find(self, a):
        while self.p[a] != a:
            self.p[a] = self.p[self.p[a]]; a = self.p[a]
        return a
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb: self.p[rb] = ra

def near_duplicate_groups(hexes, thumbs, thr, pix_thr, chunk=128):
    # Tahap 1: kandidat via jarak Hamming dHash. Tahap 2: konfirmasi via selisih piksel thumbnail 32x32.
    bits = np.unpackbits(np.stack([np.frombuffer(bytes.fromhex(h), np.uint8) for h in hexes]), axis=1).astype(bool)
    T = np.stack(thumbs).astype(np.float32)
    n = len(bits); uf = UnionFind(n); pairs = []
    for s in range(0, n, chunk):
        d = (bits[s:s + chunk, None, :] != bits[None, :, :]).sum(-1)   # jarak Hamming
        ii, jj = np.where(d <= thr)
        for i, j in zip(ii + s, jj):
            if j > i:
                pdiff = float(np.abs(T[i] - T[j]).mean())
                if pdiff < pix_thr:
                    uf.union(i, j); pairs.append((i, j, int(d[i - s, j])))
    return np.array([uf.find(i) for i in range(n)]), pairs

log = []
d = df_q.copy(); log.append(("Total awal", len(d), ""))
bad = d[~d.ok]; d = d[d.ok].copy()
log.append(("Citra rusak / gagal dibuka", -len(bad), "; ".join(bad.err.str.split(":").str[0].head(3))))
small = (d.width < MIN_SIDE) | (d.height < MIN_SIDE)
log.append((f"Resolusi < {MIN_SIDE}px", -int(small.sum()), "")); d = d[~small]

conf_md5 = d.groupby("md5").label.nunique(); conf_md5 = conf_md5[conf_md5 > 1].index
n_conf = d.md5.isin(conf_md5).sum(); d = d[~d.md5.isin(conf_md5)]
log.append(("Duplikat persis dgn label konflik", -int(n_conf), ""))
n_before = len(d); d = d.drop_duplicates("md5").reset_index(drop=True)
log.append(("Duplikat persis (MD5) disisakan satu", -(n_before - len(d)), ""))

d["group"], ND_PAIRS = near_duplicate_groups(d.dhash.tolist(), d.thumb.tolist(), NEAR_DUP_HAMMING, NEAR_DUP_PIXEL)
g_conf = d.groupby("group").label.nunique(); g_conf = g_conf[g_conf > 1].index
n_gc = d.group.isin(g_conf).sum(); d = d[~d.group.isin(g_conf)]
log.append((f"Near-duplicate (dHash<= {NEAR_DUP_HAMMING}) label konflik", -int(n_gc), ""))
grp_size = d.group.map(d.group.value_counts())
log.append(("Citra dalam kelompok near-duplicate (dipertahankan, dikunci 1 split)", int((grp_size > 1).sum()), "info"))

vc = d.label.value_counts(); rare = vc[vc < MIN_PER_CLASS].index
log.append((f"Kelas langka (< {MIN_PER_CLASS} citra): {list(rare)}", -int(d.label.isin(rare).sum()), ""))
d = d[~d.label.isin(rare)].reset_index(drop=True)
log.append(("Total bersih", len(d), ""))
if len(d) < 0.8 * len(df_q):
    print("⚠ Lebih dari 20% data terbuang - periksa ambang NEAR_DUP_* / struktur label!")

df = d
CLEAN_LOG = pd.DataFrame(log, columns=["Langkah", "Jumlah", "Keterangan"])
display(CLEAN_LOG)
REPORT["clean_log"] = CLEAN_LOG.values.tolist()
REPORT["n_clean"] = len(df)
REPORT["clean_per_class"] = df.label.value_counts().sort_index().to_dict()
REPORT["n_near_dup_pairs"] = len(ND_PAIRS)
print("Pasangan near-duplicate terdeteksi:", len(ND_PAIRS))

### 2.2 Bukti deteksi near-duplicate
Menampilkan beberapa pasangan foto yang terdeteksi sangat mirip, untuk memastikan algoritma dHash bekerja dengan benar.

In [ ]:
# Contoh pasangan near-duplicate yang terdeteksi (bukti dedup bekerja)
# gunakan tabel sesudah dedup MD5 (index sama dgn yang dipakai near_duplicate_groups)
_tmp = df_q[df_q.ok & ~((df_q.width < MIN_SIDE) | (df_q.height < MIN_SIDE))]
_tmp = _tmp[~_tmp.md5.isin(conf_md5)].drop_duplicates("md5").reset_index(drop=True)
if ND_PAIRS:
    show = ND_PAIRS[:4]
    fig, ax = plt.subplots(len(show), 2, figsize=(5.5, 2.6 * len(show)))
    ax = np.atleast_2d(ax)
    for r, (i, j, dist) in enumerate(show):
        for c, k in enumerate((i, j)):
            ax[r, c].imshow(Image.open(_tmp.path[k]).convert("RGB")); ax[r, c].axis("off")
            ax[r, c].set_title(f"{_tmp.label[k]} | Hamming={dist}", fontsize=8)
    plt.suptitle("Contoh near-duplicate terdeteksi (dHash)", y=1.0)
    plt.tight_layout(); savefig("02_near_duplicates", "Contoh pasangan near-duplicate yang dikunci pada split yang sama"); plt.show()
else:
    print("Tidak ada near-duplicate.")

## 3. Analisis & Visualisasi Dataset (EDA)

### 3.1 Distribusi kelas
Melihat jumlah citra per kelas sebelum dan sesudah pembersihan. Jika tidak seimbang, model cenderung bias ke kelas mayoritas, sehingga perlu *class weight*.

In [ ]:
CLASS_NAMES = sorted(df.label.unique()); NUM_CLASSES = len(CLASS_NAMES)
PALETTE = dict(zip(CLASS_NAMES, sns.color_palette("Set2", NUM_CLASSES)))
print("Kelas:", CLASS_NAMES)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
raw_vc = df_raw.label.value_counts().reindex(CLASS_NAMES).fillna(0)
cl_vc = df.label.value_counts().reindex(CLASS_NAMES)
x = np.arange(NUM_CLASSES)
ax[0].bar(x - 0.2, raw_vc.values, 0.4, label="sebelum cleaning", color="#bbbbbb")
ax[0].bar(x + 0.2, cl_vc.values, 0.4, label="sesudah cleaning", color=[PALETTE[c] for c in CLASS_NAMES])
for i, v in enumerate(cl_vc.values): ax[0].text(i + 0.2, v, int(v), ha="center", va="bottom", fontsize=9)
ax[0].set_xticks(x, CLASS_NAMES); ax[0].set_title("Jumlah citra per kelas"); ax[0].legend()
ax[1].pie(cl_vc.values, labels=CLASS_NAMES, autopct="%1.1f%%", colors=[PALETTE[c] for c in CLASS_NAMES], startangle=90)
ax[1].set_title("Proporsi kelas (bersih)")
plt.tight_layout(); savefig("03_class_distribution", "Distribusi kelas sebelum dan sesudah pembersihan"); plt.show()
imb = cl_vc.max() / cl_vc.min()
REPORT["imbalance_ratio"] = float(imb)
print(f"Rasio ketidakseimbangan (maks/min): {imb:.2f}")

### 3.2 Contoh citra tiap kelas
Melihat langsung seperti apa foto kulit berminyak, kering, dan normal di dataset ini.

In [ ]:
n_show = 6
fig, ax = plt.subplots(NUM_CLASSES, n_show, figsize=(2.1 * n_show, 2.2 * NUM_CLASSES))
ax = np.atleast_2d(ax)
for r, c in enumerate(CLASS_NAMES):
    sample = df[df.label == c].sample(min(n_show, (df.label == c).sum()), random_state=SEED)
    for k in range(n_show):
        ax[r, k].axis("off")
        if k < len(sample):
            ax[r, k].imshow(ImageOps.exif_transpose(Image.open(sample.path.iloc[k])).convert("RGB"))
    ax[r, 0].set_title(c.upper(), loc="left", fontsize=11, weight="bold")
plt.suptitle("Contoh citra tiap kelas", y=1.0); plt.tight_layout()
savefig("03_samples", "Contoh citra acak untuk setiap kelas"); plt.show()

### 3.3 Resolusi, rasio aspek, dan ukuran file
Ukuran citra di dataset bervariasi, sehingga semua citra perlu di-*resize* ke ukuran yang sama sebelum masuk ke CNN.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
sns.scatterplot(data=df, x="width", y="height", hue="label", palette=PALETTE, alpha=0.5, s=15, ax=ax[0])
ax[0].set_title("Resolusi asli (px)")
sns.histplot(data=df, x=df.width / df.height, hue="label", palette=PALETTE, bins=30, ax=ax[1], element="step")
ax[1].set_title("Rasio aspek (w/h)"); ax[1].set_xlabel("w/h")
sns.histplot(data=df, x="kb", hue="label", palette=PALETTE, bins=40, ax=ax[2], element="step", log_scale=True)
ax[2].set_title("Ukuran file (KB)")
plt.tight_layout(); savefig("03_resolution", "Sebaran resolusi, rasio aspek, dan ukuran file"); plt.show()
display(df.groupby("label")[["width", "height", "kb"]].describe().T.round(1))
REPORT["res_summary"] = f"lebar {df.width.min()}-{df.width.max()} px (median {df.width.median():.0f}), tinggi {df.height.min()}-{df.height.max()} px (median {df.height.median():.0f})"

### 3.4 Statistik piksel per kelas
Membandingkan kecerahan, kontras, ketajaman, dan rata-rata warna RGB antar kelas. Jika satu kelas jauh lebih terang/gelap, model bisa "curang" belajar dari pencahayaan, bukan dari kulit. Ini diantisipasi dengan augmentasi kecerahan/kontras.

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(17, 4))
for a, col, t in zip(ax, ["brightness", "contrast", "sharpness", None], ["Kecerahan (mean piksel)", "Kontras (std piksel)", "Ketajaman (var Laplacian)", None]):
    if col is None: break
    sns.boxplot(data=df, x="label", y=col, palette=PALETTE, ax=a, order=CLASS_NAMES); a.set_title(t); a.set_xlabel("")
    if col == "sharpness": a.set_yscale("log")
m = df.groupby("label")[["R", "G", "B"]].mean().reindex(CLASS_NAMES)
m.plot(kind="bar", color=["#d62728", "#2ca02c", "#1f77b4"], ax=ax[3], rot=0); ax[3].set_title("Rata-rata kanal RGB"); ax[3].set_xlabel("")
plt.tight_layout(); savefig("03_pixel_stats", "Statistik piksel per kelas: kecerahan, kontras, ketajaman, RGB"); plt.show()

### 3.5 Citra rata-rata dan histogram warna
Rata-rata ratusan foto per kelas dan distribusi warnanya memperlihatkan pola visual umum tiap kelas.

In [ ]:
# Citra rata-rata per kelas & histogram warna -> apakah ada pola visual yang membedakan kelas?
def load_small(p, s=96):
    return np.asarray(ImageOps.exif_transpose(Image.open(p)).convert("RGB").resize((s, s), Image.BILINEAR), dtype=np.float32)

fig, ax = plt.subplots(2, NUM_CLASSES, figsize=(3.6 * NUM_CLASSES, 6.5))
ax = ax.reshape(2, -1)
for k, c in enumerate(CLASS_NAMES):
    paths = df[df.label == c].sample(min(200, (df.label == c).sum()), random_state=SEED).path
    arr = np.stack([load_small(p) for p in paths])
    ax[0, k].imshow(arr.mean(0).astype(np.uint8)); ax[0, k].axis("off"); ax[0, k].set_title(f"Rata-rata: {c}")
    for ch, col in enumerate(["r", "g", "b"]):
        ax[1, k].hist(arr[..., ch].ravel(), bins=64, color=col, alpha=0.4, density=True)
    ax[1, k].set_title(f"Histogram RGB: {c}"); ax[1, k].set_yticks([])
plt.tight_layout(); savefig("03_mean_images", "Citra rata-rata dan histogram warna per kelas"); plt.show()

**Catatan EDA (diisi/diperbarui setelah melihat grafik):**
- Distribusi kelas & rasio ketidakseimbangan → menentukan perlunya *class weight*.
- Resolusi bervariasi → semua citra di-*resize* ke `IMG_SIZE`×`IMG_SIZE`.
- Perbedaan kecerahan/ketajaman antar kelas bisa menjadi *shortcut* (model belajar kondisi foto, bukan kulit) → diatasi dengan augmentasi brightness/contrast dan uji robustness.

## 4. Prapemrosesan Data
1. **Split stratified-group 72/14/14** (`StratifiedGroupKFold`): proporsi kelas sama di tiap split dan
   setiap kelompok near-duplicate berada di **satu** split saja → tidak ada kebocoran data.
2. **Decode + resize** ke 224×224 (antialias), disimpan sebagai `uint8` di cache `tf.data` (hemat RAM).
3. **Normalisasi** piksel 0–255 → 0–1 dilakukan **di dalam model** (`Rescaling`) sehingga aplikasi deployment cukup mengirim citra mentah.
4. **Augmentasi (hanya data train, on-the-fly):** flip horizontal, rotasi ±8%, zoom ±15%, translasi ±8%, kontras ±20%, kecerahan ±15%.
5. **Class weight** (`balanced`) untuk mengimbangi kelas yang lebih sedikit.
6. **One-hot + label smoothing 0.1** pada loss untuk mengurangi *overconfidence*.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight

def strat_group_split(frame, frac, seed):
    k = max(2, int(round(1 / frac)))
    sgkf = StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed)
    a, b = next(sgkf.split(frame, frame.label, groups=frame.group))
    return frame.iloc[a].reset_index(drop=True), frame.iloc[b].reset_index(drop=True)

trainval_df, test_df = strat_group_split(df, TEST_FRAC, SEED)
train_df, val_df = strat_group_split(trainval_df, VAL_FRAC, SEED)
for name, f in (("train", train_df), ("val", val_df), ("test", test_df)):
    f["split"] = name
split_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
split_df[["path", "label", "split", "group", "md5"]].to_csv(OUT_DIR / "split.csv", index=False)

# verifikasi tidak ada kebocoran
leak_g = set(train_df.group) & set(test_df.group) | set(train_df.group) & set(val_df.group) | set(val_df.group) & set(test_df.group)
leak_m = set(train_df.md5) & set(test_df.md5)
print("Kelompok near-duplicate bocor antar split:", len(leak_g), "| MD5 bocor train-test:", len(leak_m))

tab = pd.crosstab(split_df.label, split_df.split)[["train", "val", "test"]]
tab.loc["TOTAL"] = tab.sum()
display(tab)
REPORT["split_table"] = tab
REPORT["leakage"] = (len(leak_g), len(leak_m))

ax = (pd.crosstab(split_df.split, split_df.label, normalize="index").loc[["train", "val", "test"]] * 100).plot(
    kind="barh", stacked=True, color=[PALETTE[c] for c in CLASS_NAMES], figsize=(8, 2.8))
ax.set_xlabel("% citra"); ax.set_title("Komposisi kelas tiap split (stratified)"); ax.legend(bbox_to_anchor=(1, 1))
plt.tight_layout(); savefig("04_split", "Komposisi kelas pada split train/val/test"); plt.show()

cw = compute_class_weight("balanced", classes=np.array(CLASS_NAMES), y=train_df.label)
CLASS_WEIGHT = {i: float(w) for i, w in enumerate(cw)}
REPORT["class_weight"] = {CLASS_NAMES[i]: round(w, 3) for i, w in CLASS_WEIGHT.items()}
print("Class weight:", REPORT["class_weight"])

### 4.1 Pipeline data `tf.data`
Setiap citra dibaca, di-*decode*, di-*resize* ke 224×224, lalu disimpan di cache. Data train diacak setiap epoch dan diberi augmentasi acak; data validasi/test tidak diaugmentasi. Label diubah menjadi one-hot.

In [ ]:
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}

def load_image(path, label):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE], antialias=True)
    img = tf.cast(tf.clip_by_value(tf.round(img), 0, 255), tf.uint8)
    return img, label

augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08, fill_mode="reflect"),
    layers.RandomZoom(0.15, fill_mode="reflect"),
    layers.RandomTranslation(0.08, 0.08, fill_mode="reflect"),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.15, value_range=(0, 255)),
], name="augmentasi")

def make_ds(frame, training):
    ds = tf.data.Dataset.from_tensor_slices((frame.path.astype(str).tolist(), frame.label.map(CLASS_TO_IDX).values.astype("int32")))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE).cache()
    if training:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE)
    if training:
        ds = ds.map(lambda x, y: (augment(tf.cast(x, tf.float32), training=True), y), num_parallel_calls=AUTOTUNE)
    else:
        ds = ds.map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda x, y: (tf.clip_by_value(x, 0., 255.), tf.one_hot(y, NUM_CLASSES)), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

train_ds, val_ds, test_ds = make_ds(train_df, True), make_ds(val_df, False), make_ds(test_df, False)
xb, yb = next(iter(train_ds))
print("Batch train:", xb.shape, xb.dtype, "range", float(tf.reduce_min(xb)), "-", float(tf.reduce_max(xb)), "| label", yb.shape)

### 4.2 Visualisasi prapemrosesan & augmentasi
Membandingkan foto asli, hasil resize, dan beberapa variasi augmentasi. Augmentasi membuat model melihat variasi foto yang lebih banyak, sehingga lebih tahan terhadap foto nyata (robust).

In [ ]:
# Visualisasi: sebelum vs sesudah resize, dan variasi augmentasi
p0 = train_df.path.iloc[0]
orig = ImageOps.exif_transpose(Image.open(p0)).convert("RGB")
img0, _ = load_image(p0, 0)
fig, ax = plt.subplots(2, 5, figsize=(14, 6))
ax[0, 0].imshow(orig); ax[0, 0].set_title(f"Asli {orig.size[0]}x{orig.size[1]}")
ax[0, 1].imshow(img0.numpy()); ax[0, 1].set_title(f"Resize {IMG_SIZE}x{IMG_SIZE}")
x = tf.cast(img0, tf.float32)[None]
for k, a in enumerate(ax.ravel()[2:]):
    a.imshow(np.clip(augment(x, training=True)[0].numpy(), 0, 255).astype("uint8")); a.set_title(f"Augmentasi #{k + 1}")
for a in ax.ravel(): a.axis("off")
plt.suptitle(f"Prapemrosesan & augmentasi (kelas: {train_df.label.iloc[0]})"); plt.tight_layout()
savefig("04_augmentation", "Citra asli, hasil resize, dan contoh augmentasi on-the-fly"); plt.show()

## 5. Arsitektur Custom CNN
Dibangun dari nol (bobot acak, **tanpa** transfer learning):

```
Input 224x224x3 (0-255) → Rescaling(1/255)
Stem : Conv 5x5/2, 32 → BN → ReLU                         (112x112)
Blok 1-4 (residual): [Conv3x3 → BN → ReLU → Conv3x3 → BN] + shortcut(1x1 bila perlu) → ReLU
                     → MaxPool 2x2 → SpatialDropout       (56 → 28 → 14 → 7)
Head : GlobalAveragePooling → Dense 256 ReLU → Dropout 0.4 → Dense(n_kelas, softmax)
```
Alasan desain:
- **Residual shortcut** → gradien mengalir lebih baik sehingga jaringan lebih dalam tetap stabil dilatih dari nol.
- **BatchNorm** → training lebih cepat & stabil; **L2 + SpatialDropout + Dropout** → mengurangi overfitting pada dataset kecil.
- **GlobalAveragePooling** (bukan Flatten) → parameter jauh lebih sedikit, lebih tahan overfitting.
- Optimizer **Adam**, loss **categorical cross-entropy + label smoothing**, callback **EarlyStopping**, **ReduceLROnPlateau**, **ModelCheckpoint**.

In [ ]:
def conv_bn_relu(x, f, k=3, s=1, l2=1e-4, name=None):
    x = layers.Conv2D(f, k, strides=s, padding="same", use_bias=False, kernel_regularizer=regularizers.l2(l2))(x)
    x = layers.BatchNormalization()(x)
    return layers.Activation("relu", name=name)(x)

def res_block(x, f, drop, l2, idx):
    sc = x
    y = conv_bn_relu(x, f, l2=l2)
    y = layers.Conv2D(f, 3, padding="same", use_bias=False, kernel_regularizer=regularizers.l2(l2))(y)
    y = layers.BatchNormalization()(y)
    if sc.shape[-1] != f:
        sc = layers.Conv2D(f, 1, use_bias=False, kernel_regularizer=regularizers.l2(l2))(sc)
        sc = layers.BatchNormalization()(sc)
    y = layers.Activation("relu", name=f"block{idx}_out")(layers.Add()([y, sc]))
    y = layers.MaxPooling2D(2)(y)
    return layers.SpatialDropout2D(drop)(y)

def build_model(filters=(32, 64, 128, 256), dense=256, dropout=0.4, l2=1e-4, lr=1e-3, **_):
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image")
    x = layers.Rescaling(1 / 255.0, name="rescale")(inp)
    x = conv_bn_relu(x, filters[0], k=5, s=2, l2=l2, name="stem")
    for i, f in enumerate(filters, start=1):
        x = res_block(x, f, drop=min(0.1 + 0.05 * (i - 1), 0.3), l2=l2, idx=i)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dense(dense, activation="relu", kernel_regularizer=regularizers.l2(l2))(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax", name="pred", dtype="float32")(x)
    m = keras.Model(inp, out, name="SkinTypeCNN")
    m.compile(optimizer=keras.optimizers.Adam(lr),
              loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH),
              metrics=["accuracy", keras.metrics.AUC(name="auc")])
    return m

def layer_table(m):
    rows = []
    for l in m.layers:
        try: shp = tuple(l.output.shape)
        except Exception: shp = "-"
        rows.append((l.name, l.__class__.__name__, str(shp), l.count_params()))
    return pd.DataFrame(rows, columns=["Layer", "Tipe", "Output shape", "Param"])

LAST_CONV = f"block{len(HP['filters'])}_out"

## 5b. (Bonus) Optimisasi Hiperparameter — Random Search
Aktifkan dengan `RUN_HPO = True`. Mencoba `HPO_TRIALS` kombinasi acak (learning rate, dropout, ukuran dense,
lebar filter, L2) masing-masing `HPO_EPOCHS` epoch, dipilih berdasarkan **val_accuracy** terbaik (data test tidak disentuh).
Konfigurasi terbaik otomatis dipakai untuk training final.

In [ ]:
SEARCH_SPACE = dict(lr=[1e-3, 5e-4, 3e-4], dropout=[0.3, 0.4, 0.5], dense=[128, 256],
                    filters=[(32, 64, 128, 256), (24, 48, 96, 192), (32, 64, 128, 192)], l2=[1e-4, 5e-4])
HPO_RESULTS = None
if RUN_HPO:
    rng = random.Random(SEED); trials, tried = [], set()
    while len(trials) < HPO_TRIALS:
        cfg = {k: rng.choice(v) for k, v in SEARCH_SPACE.items()}
        key = json.dumps(cfg, sort_keys=True, default=str)
        if key not in tried: tried.add(key); trials.append(cfg)
    rows = []
    for t, cfg in enumerate(trials, 1):
        keras.backend.clear_session(); keras.utils.set_random_seed(SEED)
        m = build_model(**cfg)
        h = m.fit(train_ds, validation_data=val_ds, epochs=HPO_EPOCHS, class_weight=CLASS_WEIGHT, verbose=0,
                  callbacks=[keras.callbacks.EarlyStopping("val_accuracy", patience=5, restore_best_weights=True, mode="max")])
        best = max(h.history["val_accuracy"])
        rows.append({"trial": t - 1, **{k: str(v) for k, v in cfg.items()}, "params": m.count_params(), "best_val_acc": best})
        print(f"Trial {t}/{len(trials)} {cfg} -> val_acc={best:.4f}")
    HPO_RESULTS = pd.DataFrame(rows).sort_values("best_val_acc", ascending=False).reset_index(drop=True)
    display(HPO_RESULTS)
    HP.update(trials[int(HPO_RESULTS.loc[0, "trial"])]); print("Hiperparameter terbaik:", HP)
    plt.figure(figsize=(8, 3)); plt.barh(range(len(HPO_RESULTS)), HPO_RESULTS.best_val_acc, color="#4c72b0")
    plt.yticks(range(len(HPO_RESULTS)), [f"lr={r.lr}, do={r.dropout}, d={r.dense}, f={r.filters}" for r in HPO_RESULTS.itertuples()], fontsize=7)
    plt.gca().invert_yaxis(); plt.xlabel("best val_accuracy"); plt.title("Hasil random search")
    plt.tight_layout(); savefig("05_hpo", "Hasil random search hiperparameter"); plt.show()
    REPORT["hpo"] = HPO_RESULTS
else:
    print("RUN_HPO=False -> memakai hiperparameter default:", HP)

### 5.0 Membangun model final
Model dibangun dengan hiperparameter `HP` (default, atau hasil HPO bila diaktifkan). `model.summary()` menampilkan setiap layer, ukuran output, dan jumlah parameter.

In [ ]:
keras.backend.clear_session(); keras.utils.set_random_seed(SEED)
model = build_model(**HP)
LAST_CONV = f"block{len(HP['filters'])}_out"
LAYER_TABLE = layer_table(model)
model.summary(line_length=110)
REPORT["params"] = int(model.count_params())
REPORT["hp"] = {k: str(v) for k, v in HP.items()}
try:
    keras.utils.plot_model(model, to_file=str(FIG_DIR / "05_model_graph.png"), show_shapes=True, dpi=70)
except Exception as e:
    print("plot_model dilewati:", e)

### 5.1 Training

In [ ]:
CKPT = MODEL_DIR / "best_checkpoint.keras"
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=12, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6, verbose=1),
    keras.callbacks.ModelCheckpoint(str(CKPT), monitor="val_loss", save_best_only=True),
    keras.callbacks.CSVLogger(str(OUT_DIR / "training_log.csv")),
]
t0 = time.time()
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, class_weight=CLASS_WEIGHT, callbacks=callbacks, verbose=1)
TRAIN_MIN = (time.time() - t0) / 60
H = pd.DataFrame(history.history); H.index += 1
best_ep = int(H.val_loss.idxmin())
REPORT["train"] = dict(minutes=round(TRAIN_MIN, 1), epochs_run=len(H), best_epoch=best_ep,
                       train_acc=float(H.accuracy[best_ep]), val_acc=float(H.val_accuracy[best_ep]),
                       train_loss=float(H.loss[best_ep]), val_loss=float(H.val_loss[best_ep]))
print(REPORT["train"])

### 5.2 Kurva training
Grafik akurasi dan loss train vs validasi per epoch. Jika kurva validasi mengikuti kurva train, model tidak overfitting. Garis putus-putus menandai epoch terbaik yang bobotnya dipakai. Grafik ketiga menunjukkan learning rate yang diturunkan otomatis oleh ReduceLROnPlateau.

In [ ]:
lr_col = "learning_rate" if "learning_rate" in H else ("lr" if "lr" in H else None)
fig, ax = plt.subplots(1, 3 if lr_col else 2, figsize=(16, 4.2))
ax[0].plot(H.accuracy, label="train"); ax[0].plot(H.val_accuracy, label="validasi")
ax[0].axvline(best_ep, ls="--", c="gray", lw=1, label=f"best epoch {best_ep}"); ax[0].set_title("Akurasi"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(H.loss, label="train"); ax[1].plot(H.val_loss, label="validasi")
ax[1].axvline(best_ep, ls="--", c="gray", lw=1); ax[1].set_title("Loss"); ax[1].set_xlabel("epoch"); ax[1].legend()
if lr_col:
    ax[2].plot(H[lr_col], c="purple"); ax[2].set_yscale("log"); ax[2].set_title("Learning rate (ReduceLROnPlateau)"); ax[2].set_xlabel("epoch")
plt.tight_layout(); savefig("05_training_curves", "Kurva akurasi, loss, dan learning rate selama training"); plt.show()

## 6. Evaluasi pada Data Test
Data test **tidak pernah dilihat** selama training/HPO. Dilaporkan: accuracy, precision/recall/F1 per kelas, macro-F1,
confusion matrix, ROC-AUC, serta **Test-Time Augmentation (TTA)** sederhana (rata-rata prediksi citra asli + flip horizontal).

In [ ]:
from sklearn.metrics import (accuracy_score, f1_score, classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, precision_score, recall_score)
from sklearn.preprocessing import label_binarize

X_test = np.concatenate([x.numpy() for x, _ in test_ds]).astype("float32")
y_test = test_df.label.map(CLASS_TO_IDX).values
test_loss, test_acc, test_auc = model.evaluate(test_ds, verbose=0)
P = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
P_tta = (P + model.predict(np.ascontiguousarray(X_test[:, :, ::-1, :]), batch_size=BATCH_SIZE, verbose=0)) / 2
y_pred, y_tta = P.argmax(1), P_tta.argmax(1)

def metrics(y, yp, prob):
    yb = label_binarize(y, classes=range(NUM_CLASSES))
    auc = roc_auc_score(yb, prob, average="macro", multi_class="ovr") if NUM_CLASSES > 2 else roc_auc_score(y, prob[:, 1])
    return dict(accuracy=accuracy_score(y, yp), precision_macro=precision_score(y, yp, average="macro"),
                recall_macro=recall_score(y, yp, average="macro"), f1_macro=f1_score(y, yp, average="macro"), auc_macro=auc)

M = pd.DataFrame({"Tanpa TTA": metrics(y_test, y_pred, P), "Dengan TTA (flip)": metrics(y_test, y_tta, P_tta)}).T.round(4)
M["loss"] = [round(test_loss, 4), np.nan]
display(M)
REPORT["test_metrics"] = M
rep = pd.DataFrame(classification_report(y_test, y_pred, target_names=CLASS_NAMES, output_dict=True)).T.round(3)
display(rep); REPORT["class_report"] = rep

### 6.0 Confusion matrix & kurva ROC
Confusion matrix menunjukkan kelas mana yang sering tertukar. Kurva ROC/AUC mengukur kemampuan model memisahkan tiap kelas dari kelas lain (AUC 1 = sempurna, 0,5 = menebak acak).

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(1, 3, figsize=(17, 4.6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax[0], cbar=False)
ax[0].set_title("Confusion matrix (jumlah)"); ax[0].set_xlabel("Prediksi"); ax[0].set_ylabel("Aktual")
sns.heatmap(cm / cm.sum(1, keepdims=True), annot=True, fmt=".2f", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax[1], cbar=False, vmin=0, vmax=1)
ax[1].set_title("Confusion matrix (dinormalisasi per baris = recall)"); ax[1].set_xlabel("Prediksi"); ax[1].set_ylabel("Aktual")
yb = label_binarize(y_test, classes=range(NUM_CLASSES))
for k, c in enumerate(CLASS_NAMES):
    col = k if NUM_CLASSES > 2 else 1
    yy = yb[:, k] if NUM_CLASSES > 2 else (y_test == 1)
    fpr, tpr, _ = roc_curve(yy, P[:, col]); ax[2].plot(fpr, tpr, label=f"{c} (AUC={roc_auc_score(yy, P[:, col]):.3f})", color=PALETTE[c])
    if NUM_CLASSES == 2: break
ax[2].plot([0, 1], [0, 1], "k--", lw=0.8); ax[2].set_title("ROC one-vs-rest"); ax[2].set_xlabel("FPR"); ax[2].set_ylabel("TPR"); ax[2].legend()
plt.tight_layout(); savefig("06_confusion_roc", "Confusion matrix dan kurva ROC pada data test"); plt.show()

### 6.0b Analisis confidence & kesalahan
Model yang baik yakin saat benar dan ragu saat salah. Ditampilkan juga contoh kesalahan dengan confidence tertinggi untuk dianalisis.

In [ ]:
conf = P.max(1); correct = y_pred == y_test
fig, ax = plt.subplots(1, 1, figsize=(7, 3.5))
ax.hist(conf[correct], bins=20, alpha=0.7, label="benar", color="#2ca02c"); ax.hist(conf[~correct], bins=20, alpha=0.7, label="salah", color="#d62728")
ax.set_xlabel("confidence (prob. maksimum)"); ax.set_title("Distribusi confidence prediksi test"); ax.legend()
plt.tight_layout(); savefig("06_confidence", "Distribusi confidence untuk prediksi benar vs salah"); plt.show()

wrong = np.where(~correct)[0]
wrong = wrong[np.argsort(-conf[wrong])][:8]
if len(wrong):
    fig, ax = plt.subplots(1, len(wrong), figsize=(2.3 * len(wrong), 2.8))
    for a, i in zip(np.atleast_1d(ax), wrong):
        a.imshow(X_test[i].astype("uint8")); a.axis("off")
        a.set_title(f"asli: {CLASS_NAMES[y_test[i]]}\npred: {CLASS_NAMES[y_pred[i]]} ({conf[i]:.2f})", fontsize=8, color="crimson")
    plt.suptitle("Kesalahan dengan confidence tertinggi", y=1.04); savefig("06_misclassified", "Contoh salah klasifikasi dengan confidence tertinggi"); plt.show()

### 6.1 Uji Ketangguhan (Robustness)
Model diuji pada data test yang **sengaja dirusak** (simulasi foto pengguna asli: gelap/terang, blur, noise sensor,
kompresi JPEG WhatsApp, rotasi). Penurunan akurasi yang kecil = model tangguh (bobot rubrik 60% menekankan *robust*).

In [ ]:
def perturb(X, kind):
    out = []
    for x in X.astype("uint8"):
        if kind == "gelap (-40)":      y = np.clip(x.astype(int) - 40, 0, 255)
        elif kind == "terang (+40)":   y = np.clip(x.astype(int) + 40, 0, 255)
        elif kind == "blur (5x5)":     y = cv2.GaussianBlur(x, (5, 5), 0)
        elif kind == "noise (σ=15)":   y = np.clip(x + np.random.default_rng(0).normal(0, 15, x.shape), 0, 255)
        elif kind == "JPEG q=30":
            ok, enc = cv2.imencode(".jpg", x[..., ::-1], [cv2.IMWRITE_JPEG_QUALITY, 30]); y = cv2.imdecode(enc, 1)[..., ::-1]
        elif kind == "rotasi 15°":
            Mx = cv2.getRotationMatrix2D((IMG_SIZE / 2, IMG_SIZE / 2), 15, 1); y = cv2.warpAffine(x, Mx, (IMG_SIZE, IMG_SIZE), borderMode=cv2.BORDER_REFLECT)
        elif kind == "flip horizontal": y = x[:, ::-1]
        else: y = x
        out.append(np.ascontiguousarray(y, dtype="float32"))
    return np.stack(out)

KINDS = ["asli", "gelap (-40)", "terang (+40)", "blur (5x5)", "noise (σ=15)", "JPEG q=30", "rotasi 15°", "flip horizontal"]
rob = []
for k in KINDS:
    pk = model.predict(perturb(X_test, k), batch_size=BATCH_SIZE, verbose=0).argmax(1)
    rob.append((k, accuracy_score(y_test, pk), f1_score(y_test, pk, average="macro")))
ROB = pd.DataFrame(rob, columns=["Perturbasi", "Accuracy", "Macro-F1"]).round(4)
ROB["Δ acc vs asli"] = (ROB.Accuracy - ROB.Accuracy[0]).round(4)
display(ROB); REPORT["robustness"] = ROB

plt.figure(figsize=(9, 3.6))
bars = plt.bar(ROB.Perturbasi, ROB.Accuracy, color=["#4c72b0"] + ["#dd8452"] * (len(ROB) - 1))
for b, v in zip(bars, ROB.Accuracy): plt.text(b.get_x() + b.get_width() / 2, v, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
plt.ylim(0, 1.05); plt.xticks(rotation=25, ha="right"); plt.ylabel("accuracy"); plt.title("Robustness: akurasi test pada citra terdistorsi")
plt.tight_layout(); savefig("06_robustness", "Akurasi model pada data test yang diberi gangguan"); plt.show()

### 6.2 Grad-CAM — bagian wajah mana yang dilihat model?
Grad-CAM menghitung gradien skor kelas terhadap feature map konvolusi terakhir untuk membuat *heatmap*.
Jika area panas berada di kulit (dahi, pipi, hidung — zona T), model belajar fitur yang benar, bukan latar belakang.

In [ ]:
grad_model = keras.Model(model.inputs, [model.get_layer(LAST_CONV).output, model.output])

def grad_cam(x):
    x = tf.convert_to_tensor(x[None], tf.float32)
    with tf.GradientTape() as tape:
        conv, pred = grad_model(x, training=False)
        idx = int(tf.argmax(pred[0])); score = pred[:, idx]
    g = tape.gradient(score, conv)
    w = tf.reduce_mean(g, axis=(0, 1, 2))
    cam = tf.nn.relu(tf.reduce_sum(conv[0] * w, axis=-1)).numpy()
    cam = cam / (cam.max() + 1e-8)
    return cv2.resize(cam, (IMG_SIZE, IMG_SIZE)), idx, float(pred[0, idx])

rng = np.random.default_rng(SEED)
idxs = np.concatenate([rng.choice(np.where(y_test == k)[0], min(2, (y_test == k).sum()), replace=False) for k in range(NUM_CLASSES)])
fig, ax = plt.subplots(2, len(idxs), figsize=(2.4 * len(idxs), 5))
for c, i in enumerate(idxs):
    cam, pi, pc = grad_cam(X_test[i])
    ax[0, c].imshow(X_test[i].astype("uint8")); ax[0, c].set_title(f"asli: {CLASS_NAMES[y_test[i]]}", fontsize=8)
    ax[1, c].imshow(X_test[i].astype("uint8")); ax[1, c].imshow(cam, cmap="jet", alpha=0.45)
    ax[1, c].set_title(f"pred: {CLASS_NAMES[pi]} ({pc:.2f})", fontsize=8, color="green" if pi == y_test[i] else "crimson")
for a in ax.ravel(): a.axis("off")
plt.suptitle("Grad-CAM pada data test"); plt.tight_layout(); savefig("06_gradcam", "Grad-CAM: area citra yang paling memengaruhi prediksi"); plt.show()

## 7. Ekspor Model & Simulasi Inference (dipakai aplikasi deployment)
Model disimpan sebagai `.keras` (+ `labels.json`). Fungsi `predict_image` di bawah **sama persis** dengan logika di `app.py`.

In [ ]:
RECOMMENDATIONS = {
    "oily":   dict(judul="Kulit Berminyak", ciri="Produksi sebum tinggi, wajah mengkilap terutama zona T, pori besar, rentan komedo/jerawat.",
                   rutin=["Gentle foaming / gel cleanser 2x sehari", "Toner/exfoliant BHA (salicylic acid) 2-3x seminggu",
                          "Serum niacinamide untuk kontrol minyak & pori", "Moisturizer gel oil-free, non-comedogenic",
                          "Sunscreen tekstur gel/fluid, matte finish"],
                   hindari=["Produk berbasis minyak berat / occlusive tebal", "Scrub kasar berlebihan", "Over-cleansing (memicu minyak berlebih)"]),
    "dry":    dict(judul="Kulit Kering", ciri="Terasa kencang, kusam, bersisik/mengelupas, garis halus lebih terlihat.",
                   rutin=["Cleanser creamy / low-pH tanpa SLS", "Hydrating toner/essence (hyaluronic acid)",
                          "Serum ceramide / panthenol untuk skin barrier", "Moisturizer krim kaya (ceramide, shea butter, squalane)",
                          "Sunscreen moisturizing"],
                   hindari=["Toner beralkohol tinggi", "Cleanser busa kuat", "Eksfoliasi terlalu sering", "Air terlalu panas"]),
    "normal": dict(judul="Kulit Normal", ciri="Seimbang, tidak terlalu berminyak/kering, pori kecil, jarang bermasalah.",
                   rutin=["Gentle cleanser", "Moisturizer ringan (lotion)", "Antioksidan (vitamin C) pagi hari",
                          "Sunscreen SPF 30+ setiap hari", "Eksfoliasi ringan AHA 1x seminggu (opsional)"],
                   hindari=["Mengganti-ganti banyak produk sekaligus", "Melewatkan sunscreen"]),
    "combination": dict(judul="Kulit Kombinasi", ciri="Zona T berminyak, pipi normal/kering.",
                   rutin=["Gentle gel cleanser", "Niacinamide", "Moisturizer ringan; krim lebih kaya di area kering", "Sunscreen ringan"],
                   hindari=["Produk terlalu berat di zona T"]),
    "sensitive": dict(judul="Kulit Sensitif", ciri="Mudah kemerahan, perih, atau iritasi.",
                   rutin=["Cleanser fragrance-free", "Moisturizer dengan centella/ceramide", "Sunscreen mineral (zinc oxide)"],
                   hindari=["Pewangi, alkohol, eksfoliasi keras", "Patch test dulu sebelum produk baru"]),
}

def preprocess_pil(img):
    img = ImageOps.exif_transpose(img).convert("RGB")
    x = tf.image.resize(np.asarray(img), [IMG_SIZE, IMG_SIZE], antialias=True)
    return tf.clip_by_value(tf.round(x), 0, 255).numpy().astype("float32")

def predict_image(img, tta=True):
    x = preprocess_pil(img)[None]
    p = model.predict(x, verbose=0)[0]
    if tta: p = (p + model.predict(np.ascontiguousarray(x[:, :, ::-1, :]), verbose=0)[0]) / 2
    return {c: float(p[i]) for i, c in enumerate(CLASS_NAMES)}

model.save(MODEL_DIR / "skin_type_cnn.keras")
meta = dict(class_names=CLASS_NAMES, img_size=IMG_SIZE, dataset=DATASET_USED, created=datetime.now().isoformat(timespec="seconds"),
            test_accuracy=float(M.loc["Tanpa TTA", "accuracy"]), test_accuracy_tta=float(M.loc["Dengan TTA (flip)", "accuracy"]),
            hp=REPORT["hp"], tf_version=tf.__version__, keras_version=keras.__version__)
json.dump(meta, open(MODEL_DIR / "labels.json", "w"), indent=2)
try:
    conv = tf.lite.TFLiteConverter.from_keras_model(model); conv.optimizations = [tf.lite.Optimize.DEFAULT]
    (MODEL_DIR / "skin_type_cnn.tflite").write_bytes(conv.convert())
except Exception as e:
    print("Konversi TFLite dilewati:", e)
for f in sorted(MODEL_DIR.iterdir()): print(f"{f.name:28s} {f.stat().st_size / 1e6:6.2f} MB")

# uji: muat ulang model dari file (memastikan file deployment valid) lalu prediksi beberapa citra test
reloaded = keras.models.load_model(MODEL_DIR / "skin_type_cnn.keras")
assert np.allclose(reloaded.predict(X_test[:4], verbose=0), model.predict(X_test[:4], verbose=0), atol=1e-4)
print("✓ Model tersimpan & dimuat ulang dengan hasil identik")

demo = test_df.groupby("label").head(1)
fig, ax = plt.subplots(1, len(demo), figsize=(4 * len(demo), 4.4))
for a, (_, r) in zip(np.atleast_1d(ax), demo.iterrows()):
    pr = predict_image(Image.open(r.path)); top = max(pr, key=pr.get)
    a.imshow(ImageOps.exif_transpose(Image.open(r.path)).convert("RGB")); a.axis("off")
    a.set_title(f"asli: {r.label} | pred: {top} ({pr[top]:.0%})\n" + RECOMMENDATIONS.get(top, {}).get("judul", top), fontsize=9)
plt.tight_layout(); savefig("07_inference_demo", "Simulasi inference end-to-end (fungsi yang sama dengan aplikasi web)"); plt.show()

## 8. Laporan Otomatis (PDF)
Sel ini merangkum semua angka & grafik di atas menjadi `PA_outputs/Laporan_PA_KB.pdf`.

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
A4 = (8.27, 11.69)
PDF_PATH = OUT_DIR / "Laporan_PA_KB.pdf"

class Doc:
    def __init__(self, pdf): self.pdf, self.fig, self.y = pdf, None, 0
    def new(self):
        self.flush(); self.fig = plt.figure(figsize=A4); self.y = 0.95
        self.fig.text(0.5, 0.02, f"{KELOMPOK} — {JUDUL}", ha="center", fontsize=7, color="gray")
    def flush(self):
        if self.fig is not None: self.pdf.savefig(self.fig); plt.close(self.fig); self.fig = None
    def need(self, h):
        if self.fig is None or self.y - h < 0.05: self.new()
    def h1(self, t): self.need(0.06); self.y -= 0.01; self.fig.text(0.07, self.y, t, fontsize=14, weight="bold", color="#1f3b73"); self.y -= 0.035
    def h2(self, t): self.need(0.04); self.fig.text(0.07, self.y, t, fontsize=11, weight="bold"); self.y -= 0.027
    def p(self, t, size=9, mono=False, width=100, indent=0.07):
        for line in str(t).split("\n"):
            for w in (textwrap.wrap(line, width if not mono else 105) or [""]):
                self.need(0.018)
                kw = dict(family="monospace") if mono else {}
                self.fig.text(indent, self.y, w, fontsize=size if not mono else size - 1.5, **kw); self.y -= 0.0165 if not mono else 0.0145
        self.y -= 0.006
    def bullets(self, items):
        for it in items: self.p("•  " + str(it), indent=0.09)
    def table(self, frame, size=8, col_w=None):
        frame = frame.copy(); frame.index = frame.index.map(str)
        txt = frame.to_string(); self.p(txt, size=size + 1.5, mono=True)
    def image(self, name, h=0.38):
        if name not in FIGS: return
        path, cap = FIGS[name]
        img = plt.imread(path); ar = img.shape[0] / img.shape[1]
        w = min(0.86, h / ar * A4[1] / A4[0]); hh = w * ar * A4[0] / A4[1]
        self.need(hh + 0.04)
        axx = self.fig.add_axes([0.5 - w / 2, self.y - hh, w, hh]); axx.imshow(img); axx.axis("off")
        self.y -= hh + 0.005
        self.fig.text(0.5, self.y - 0.01, cap, ha="center", fontsize=7.5, style="italic", color="#444"); self.y -= 0.03

def fmt_pct(v): return "-" if v is None else f"{v * 100:.2f}%"
done = lambda k: k in REPORT

with PdfPages(PDF_PATH) as pdf:
    D = Doc(pdf)
    # ---------- Cover
    D.new(); D.y = 0.80
    D.fig.text(0.5, 0.86, "LAPORAN PROYEK AKHIR", ha="center", fontsize=18, weight="bold", color="#1f3b73")
    D.fig.text(0.5, 0.83, "Praktikum Kecerdasan Buatan", ha="center", fontsize=13)
    D.fig.text(0.5, 0.76, JUDUL, ha="center", fontsize=14, weight="bold", wrap=True)
    D.fig.text(0.5, 0.73, "Custom Convolutional Neural Network (tanpa transfer learning)", ha="center", fontsize=11, style="italic")
    D.fig.text(0.5, 0.66, KELOMPOK, ha="center", fontsize=12, weight="bold")
    yy = 0.63
    for n, nim, _ in ANGGOTA:
        D.fig.text(0.5, yy, f"{n}  —  {nim}", ha="center", fontsize=11); yy -= 0.025
    D.fig.text(0.5, yy - 0.02, f"Aslab: {ASLAB}", ha="center", fontsize=10)
    D.fig.text(0.5, 0.15, f"Dibuat otomatis dari notebook: {datetime.now():%d %B %Y, %H:%M}", ha="center", fontsize=9, color="gray")
    D.y = 0.0

    # ---------- Ringkasan
    D.h1("1. Ringkasan Progress")
    tm = REPORT.get("test_metrics")
    status = [
        ("Pemilihan & pengumpulan dataset", done("n_raw")),
        ("Pemeriksaan kualitas & pembersihan data (rusak, duplikat, near-duplicate)", done("n_clean")),
        ("Analisis & visualisasi dataset (EDA)", "03_class_distribution" in FIGS),
        ("Prapemrosesan: split stratified-group, resize, augmentasi, class weight", done("split_table")),
        ("Perancangan arsitektur custom CNN", done("params")),
        ("Training model", done("train")),
        ("Evaluasi testing + uji robustness + Grad-CAM", done("robustness")),
        ("(Bonus) Optimisasi hiperparameter random search", done("hpo")),
        ("Ekspor model untuk deployment (.keras, TFLite)", (MODEL_DIR / "skin_type_cnn.keras").exists()),
        ("Aplikasi web SkinSense (Flask) + deploy Hugging Face Spaces (bagian 9 notebook)", True),
    ]
    for s, ok in status: D.p(("[SELESAI]   " if ok else "[PROSES]     ") + s, indent=0.09)
    if tm is not None:
        D.h2("Hasil utama")
        D.bullets([f"Akurasi test: {fmt_pct(tm.iloc[0]['accuracy'])}  |  dengan TTA: {fmt_pct(tm.iloc[1]['accuracy'])}",
                   f"Macro-F1 test: {tm.iloc[0]['f1_macro']:.4f}  |  Macro ROC-AUC: {tm.iloc[0]['auc_macro']:.4f}",
                   f"Parameter model: {REPORT.get('params', 0):,} (dilatih dari nol)",
                   f"Epoch terbaik: {REPORT['train']['best_epoch']} dari {REPORT['train']['epochs_run']}  ({REPORT['train']['minutes']} menit, {REPORT.get('env', {}).get('gpu', '')})"])

    # ---------- Data
    D.h1("2. Pengumpulan Data")
    D.bullets([f"Dataset: Oily, Dry and Normal Skin Types Dataset — kaggle.com/datasets/{PRIMARY_DATASET}",
               "Dataset awal (Skin_Type_Prediction_Dataset) diganti karena berupa data sintetis tabular, bukan citra",
               f"Jumlah citra terbaca: {REPORT.get('n_raw')}  →  setelah dibersihkan: {REPORT.get('n_clean')}",
               f"Kelas: {', '.join(CLASS_NAMES)}  (rasio ketidakseimbangan {REPORT.get('imbalance_ratio', 0):.2f})",
               f"Resolusi asli: {REPORT.get('res_summary', '-')}"])
    if REPORT.get("dataset_used") != PRIMARY_DATASET:
        D.p("Catatan: dataset yang dipilih tidak berisi citra yang cukup (hasil pengecekan: "
            + "; ".join(f"{k}: {v}" for k, v in REPORT.get("dataset_check", {}).items())
            + "). Karena PA mewajibkan data citra, digunakan dataset citra jenis kulit (oily/dry/normal) sebagai pengganti — akan dikonfirmasi ke aslab.")
    D.image("03_samples", 0.36)

    D.h1("3. Prapemrosesan Data")
    D.h2("3.1 Pembersihan data")
    D.table(pd.DataFrame(REPORT.get("clean_log", []), columns=["Langkah", "Jumlah", "Keterangan"]).set_index("Langkah"))
    D.image("02_near_duplicates", 0.30)
    D.h2("3.2 Split, resize, augmentasi, class weight")
    D.bullets(["Split stratified-group ±72/14/14 (StratifiedGroupKFold) — kelompok near-duplicate dikunci di satu split",
               f"Pengecekan kebocoran: kelompok bocor = {REPORT.get('leakage', ('-', '-'))[0]}, MD5 bocor train-test = {REPORT.get('leakage', ('-', '-'))[1]}",
               f"Resize {IMG_SIZE}x{IMG_SIZE} antialias, normalisasi 1/255 di dalam model",
               "Augmentasi on-the-fly (train): flip, rotasi, zoom, translasi, kontras, kecerahan",
               f"Class weight: {REPORT.get('class_weight')}",
               f"Label smoothing {LABEL_SMOOTH}"])
    if "split_table" in REPORT: D.table(REPORT["split_table"])
    D.image("04_augmentation", 0.30)

    D.h1("4. Analisis & Visualisasi Dataset")
    for n in ("03_class_distribution", "03_resolution", "03_pixel_stats", "03_mean_images"):
        D.image(n, 0.22)

    D.h1("5. Arsitektur Model & Hiperparameter")
    D.p("Custom CNN residual 4 blok (Conv-BN-ReLU, shortcut, MaxPool, SpatialDropout) + GlobalAveragePooling + Dense + Dropout + Softmax. "
        "Loss categorical cross-entropy (label smoothing), optimizer Adam, EarlyStopping (patience 12), ReduceLROnPlateau (x0.5, patience 4), ModelCheckpoint.")
    D.table(pd.Series({**REPORT.get("hp", {}), "img_size": IMG_SIZE, "batch_size": BATCH_SIZE, "max_epochs": EPOCHS,
                       "total_params": f"{REPORT.get('params', 0):,}"}, name="nilai").to_frame())
    if "LAYER_TABLE" in globals():
        lt = LAYER_TABLE[~LAYER_TABLE.Tipe.isin(["BatchNormalization", "Activation", "Add"]) | LAYER_TABLE.Layer.str.contains("block|stem")]
        D.h2("Ringkasan layer (BN/Activation/Add disembunyikan kecuali output blok)")
        D.table(lt.set_index("Layer"))
    if "hpo" in REPORT:
        D.h2("(Bonus) Random search hiperparameter"); D.table(REPORT["hpo"]); D.image("05_hpo", 0.2)

    D.h1("6. Hasil Training")
    if "train" in REPORT:
        t = REPORT["train"]
        D.bullets([f"Epoch dijalankan: {t['epochs_run']} (early stopping), epoch terbaik: {t['best_epoch']}",
                   f"Pada epoch terbaik → train acc {fmt_pct(t['train_acc'])}, val acc {fmt_pct(t['val_acc'])}, "
                   f"train loss {t['train_loss']:.4f}, val loss {t['val_loss']:.4f}"])
    D.image("05_training_curves", 0.22)

    D.h1("7. Evaluasi Testing")
    if tm is not None: D.table(tm)
    if "class_report" in REPORT: D.table(REPORT["class_report"])
    D.image("06_confusion_roc", 0.22)
    D.image("06_confidence", 0.18)
    D.image("06_misclassified", 0.16)
    D.h2("Uji robustness")
    if "robustness" in REPORT: D.table(REPORT["robustness"].set_index("Perturbasi"))
    D.image("06_robustness", 0.22)
    D.h2("Interpretabilitas (Grad-CAM)")
    D.image("06_gradcam", 0.25)
    D.image("07_inference_demo", 0.22)

    D.h1("8. Deployment")
    D.bullets(["Model diekspor: skin_type_cnn.keras + labels.json (+ TFLite)",
               "Web app SkinSense (backend Flask + HTML/JS): upload foto / kamera → jenis kulit, probabilitas tiap kelas, rekomendasi skincare",
               "Prapemrosesan di web identik dengan training (resize 224 antialias; normalisasi di dalam model)",
               "Hosting: Hugging Face Spaces (Docker), di-deploy otomatis dari bagian 9 notebook"])

    D.h1("9. Pembagian Tugas")
    for n, nim, tugas in ANGGOTA: D.p(f"{n} ({nim}): {tugas}", indent=0.09)

    D.h1("10. Kesimpulan")
    if tm is not None:
        D.bullets([f"Custom CNN residual ({REPORT.get('params', 0):,} parameter, dilatih dari nol) mencapai akurasi test "
                   f"{fmt_pct(tm.iloc[0]['accuracy'])} (TTA {fmt_pct(tm.iloc[1]['accuracy'])}) dan macro-F1 {tm.iloc[0]['f1_macro']:.4f}.",
                   "Deduplikasi + split berbasis grup memastikan angka test tidak tergelembung oleh data leakage.",
                   "Uji robustness & Grad-CAM menunjukkan perilaku model pada foto terdistorsi dan area yang diperhatikan.",
                   "Model di-deploy sebagai web app yang memberi rekomendasi skincare sesuai jenis kulit."])
    D.flush()

print("✓ Laporan:", PDF_PATH, f"({PDF_PATH.stat().st_size / 1e6:.2f} MB)")

## 9. Deployment ke Hugging Face Spaces (link demo publik)
Sel ini men-*deploy* web app **SkinSense** (backend Flask + tampilan HTML) bersama model yang baru dilatih ke
**Hugging Face Spaces** sehingga mendapat link publik `https://huggingface.co/spaces/<username>/skinsense`.

Cara pakai:
1. Buat akun gratis di huggingface.co, lalu buka **Settings → Access Tokens → Create new token** dengan tipe **Write**.
2. Jalankan sel ini dan tempel token saat diminta (kosongkan jika ingin melewati deployment).
3. Tunggu build ±5–10 menit di halaman Space sampai statusnya **Running**.

Source code web disertakan di dalam notebook (dikompres) dan diekstrak ke folder `web/`. Versi `tensorflow` di
`requirements.txt` otomatis disamakan dengan versi TensorFlow yang dipakai training agar model pasti bisa dimuat.

In [ ]:
import base64, getpass
WEB_ZIP_B64 = "UEsDBBQAAAAIAB13aV1xI1wx3AAAABsBAAAOAAAAd2ViL0RvY2tlcmZpbGVljj1PwzAYhHf/ipNgAKmvSQABisTUpAKhNpFL+RiNYyVWXds4jkr+PWoY0Q13y/PozrBzadyj1cH6Cd5Z4zQunsauM67DSiqNbZBKDyCUXu11XEBo184tjT3KaYHWWn7JVqJeI0yp96644XlOgzUH9l6Ll/JZ4EqGwJZ184mov0cT9UG7NPD0k8CZ2G0QTIBxQ5LWgsh5UlL1mloTQfEf9Kfi4KzavKGpxevj/cNdxqqPpt5WmPdyXaIbnVE+OtAROYhSH7VsB9yCEvLrDPSFjM8pzk8ayBCK09dfUEsDBBQAAAAIAB13aV14hXyYOwQAAG8HAAANAAAAd2ViL1JFQURNRS5tZI1VTW/bRhC981c8wBdJlUlbiS9GEMCJk9hx4wqRgxzDITmSxlzubvcDjgod8h9qoJf+m/4T/5JiSdlV0h4qQIfVkjNv3rz3dIBFK3rB2vMpPnOFcw7cesF71uJxFZUEfKZbWmdZuiZrEXWILTrTsEIdfTAdXl9fY36GK67ZNeRJ41WkQDrHnPVqFTWhY72KerWiNZYmGFCgiI47ExwH3KUWUHxHAS117GgKRSpmQ5v0sjSsgyylJS+47eG1PbxRxa4TvaEWBVp2olcooI3rSI2n6V3qrKiWNKwzFVWiJJBHELJoWZGfoiGdwFTsJHPcmo51k/r4VnRNjvMsOzjAIrjYhuiysiyzO66K7OH+z4f7bw/33xIzud3gu88BKqpb1g3eKvLtaWoRKey4+wln80sUZKWwjhupQw+j/0H00uwV94GC1IXohr/m69Cpvnjo5yKNO64wurj58HPxerEo3i+mCKQtQUnlyG3ArQ/sNKnxXs0eRPEj4Juzj58u+rG/hI3lL7XWecuOPH6CooqVz2+90Ti/xOLy+nKvnuNfozjuWAefh68he7i/312dm7pltxTF+60GHTVsldnAaCWae2IT08c5FkI27azHmb0nRTodtQlcGdOiEbwzZqUYr42iCp46SwLPij3JIB9QV4lCEwl994acoJyffTEx2Bh8MZBQnmblf4xc9vso98YucyxIiUbLTSS9IbSMpVENO5RJELt6g1xmOZ5gNwJFNhiL0ePcnRmneSvy66xu0hIzKxaifSClcOj+TandhLXRO631ZL2KLWEyWYdgT4tCmZrU2vhwenJ0dDSZ5LjqzYR1D7YST2jEUkuyM1v59Eo5WPLi5ma+GPA/y3G+vx2MjPViko5OcRFXq+S0t1QzFpZq9tlx3vt+OKIiF9Pc6+HJJdWc16bQfHfo0wNTWFGyxuL8CpPJoJHJJM9mOT5ZZahJu4wuriFevmO5xCiw68jHFuVA+f9b3zjPnuW4STkUUxh49qiiqOZRNnkfgjueqE33jaBM9PrTovhhlH4MX7yIPrmr45fFC00dDeO9LDHysaH1QOkUt9TILtxQsVtGvfIyHqg+m19mW7zRjTWiA7b4wGFtGmxxxYEd6RVpbLPt4eHh0zfbonwKixJbvHtzgy0WgUL0g21SsC0DuceYi210pCHaxjAFpZMXBPYBo8Eb37GFf3rsEiq1mf+yuMGoiyqIJRfG2OKtsGpQSkcrLjFKXhtPUYZAJUblcVmUR+U4xwfWK+4qUpIs0beaojZ6mbK9F8RePk+xF8RT3FEbIkQv2bH2gm2WzR1Z7votDg4rB2OUGH1892qXAI69/MaYzZ7/9cds9hykg5CSVL+/XypDAUcP336fnZyMMZl46giWnRc/maDhnvrH1Mmz6/5/RXwi7riYnZykXVKT9IKGFHW7dB8p2iS9fmRfp8hYJfH9DVBLAwQUAAAACADBdmldqFisHMANAABMIwAACgAAAHdlYi9hcHAucHmtWltv3DqSfu9fUdBgEGlHljvJyTx4Rot1Eucy4xtiH5zBGoZQLZXUjCRSQ1KxO4H/+6JIqVt9ibMPowRutVisKha/ulEdBMFNLeQNSUMn8EALwK6DgizVRsBXksJA3TfCwgN+xSWEeW+sauHd5WUEfwFNtWpJFmgEmFrIHDUls9k/sEFZo4RG1diczAAAOtGBkMZi08CRBk3/7oWmlqQ1iX20nmZll0qyCkm3gsn1J1j0NcLS2u7k+LhROTZLZezJm/l8Ppt9EA1BqwpqIFyiEQ1IZWmhVB1DqZqCNFyfZqq3XW/NsSM8jmCJujeABUIhRrJkGPUq+3teV2ZXHWW5lElNGs1ktMEFNSb5apScBUEwE22ntAWhxjs3MtwrM97ZpSYshKzWD0RLs1KrFjq0y0YsYBi4RrucjVSyb7sVoAHZeeKyQVOPpB/4S+wkinIVOxuTsTEYkkXG9FkhNOVW6ZWffv35fJz8ucWKYv9x1ZkYfpeiIGlFKahwT8+0Vno2e3t6c5a9//wFUqdbmGWlaCjLokSTUc03CqOkQ03Szi6u3p+dT2mVSUh+E1rJpCIbBmuCIIY132MInG2DKJpdnP4r+/36/Or0fXbxFlJ4OZ+dnp9f/XH2Pjv71y2k8CNIvnZVEAN/kr/ppP98oEXnbhZtFzzNzq/+yN5dXX74/P7s8t0ZpDBP3sxnsy9n764uLs4u35/efr66vGGebn8DJZpVcAKFyG04G5H4tS/6Jg3+6XziLelWyBXWQbwmyIUWaXCtVdGzCxla9C1YIatKxIMTWWw7rKElWdWiwQ4s6d5i65D4XUmEWwgLXIoYlqLoZRVDgVUfxdApLY74DyzIoObnEtjUKIE9sVDHX0njA9pkopLurZDpXfCRpG0ISoWtkBWgxR4qaiBvCKUhDa8ewdAStZhMHq7g7LFUjUBp4e2nUwgNNiJfNSIHzEURwauj1zyZGVf9gek3pPsWpMBcSGxFQdBL23sjKGm1asCb0i2Jl3iAyYUSxvZafCftFFeiOSo1UQxSyaPcGaAiKfJDCvTS5JpIwoI0hzfbOybHZdOLAgqSFUrw4aNFaym43zBZClmgFundsLHMY4FLlKPSC9JovUVV3fRGlGBpgc2+IqxKrvsF1GhQ88SGFmKJ8iDpBcm8zwW0HP4s6QabHgxp3r+wpVbk/UQFxykK7iPPKij0LwD8T8dpD723HOQQapI5ygoMWWpwCWtN4hHCdW+wjVm0EUbU3gC8pdT0HRqPzwq1MLDEpjfgNHQLEcvDIH03gjHXhO3Ks2zUw1H3CSzKDuHm/ObA/n5aFRrZz8AqSfqYjCGZ03pjV9j0WskBsD9FaE7aw9PJ7VDaJUnVrNHadqTrHq1Ld7BArQXpX0C11qKFFVuyxhVCOMqIwSwJYdFbSzoG8++e8yZFz6LX8WmpoXaBXY3yMExv2Qbj2muUHEVQAja1WqpmiEcHEbe2P0O8ZxT0aA9SntXGhQQjdpD5ayQPiqHQ66kdSjQb6EqlW2yeR++lp9lF7w2JdoEcNa0osF5LWIzRGlrsu15C7bSdRtWactF41H5FzZbmSWiwweUzEXUMoL/AAUtDCWGjrFDy0DafSitUbQQrYKgjbQV8E5bjNbzj7NBhJeAnEXqDkZvrD/B6/hd2XIHdz+gn+zdodvrpFF5uwjiEqjNCSWyiwyjjPVULhAU6s3ZDbETNSKixEVVvfgKGhh7QcoloRq03e5+rdiEkspF+Eb48oRF7GPhfn0XXW841UIGyYomoCbkcFeBB5l3dg+GZTZ5mSq4q9jLa/2v3/+ajgY+DLhwUwmtUD74z3Ug/59lMNEE32k0BsbGmIWmEFd/oeVveeLJyz5QXfYFLqKklzRkvho60WMbeakILi0Y8G8hLjZVGmZPL1bzAqVmGUJCTtNQ0eDwGx21DtEKSxgbC70LmoB5FQT/B5DU9oORaa4h13ptpg3VXwTP3c5SV6V00bbFGsYVfn2o6tDmnK2OdPZ9ms1lBJWjKVev6HsZoVioduj4g8n2DJttrCTuFpat4HVns98Gb3z1JrLANhVE82PwoiEc73sebxd1H0Ww2yxs0hlM3mmtNzEppLzgIggtqOTv5bsh3K84ALTVY9wz/TmNHrVaGDA7p5Ob04nTciLF14jCFRSa48o8Sbm2cCF5+lgkpbJaFhpoy9rK4sxiWD2NzxM/Gyn9DtEPD7U4MLVl0t5BO5h5DcKD7CuJtkkkLFqyZi5JXMpGR0KMw1oQRKO2HRpHrkZMt/9UoDAH3lpfKflC9LFwDFO6FszKYNKA+5RTCUuusXQh48WOt7tOLBG6wERIOrMvt02QxEOyJChgH+80sbHe9SbAx8thekjRKl4164O7RltxSD0OFACOkAKxQcwL/xrUXdQwikitscJfXpgXmizGQ2BJSsOX2QzYwpK4dTRhLJtyYnLvfzNKjDaNoe5ZDNxlIoRHGhmtOd4EbySS2xHXC9izRVpkR3wlSEHJr1jiyN8XvV+qX478Zp2fm7sMpPHPVdqKh9AM2hnb4yKxDja3ZEs1zk1z10g6je+v0JJ134FB2yXfSyoThy3h7RXtfX0cxyC4pG4X29asohm+kF8pQOo94Vx9Qt0d9t/HWTlOnVU7GDP4q2moC9T/Bl49v4ei/QZMz4EI0QhJqQGkFcsjkwUXf+HRdE8yT5NWbN/x00AFCn0iF4fj68phH+Vzl/Wd4f3p+euFdY7P+R0hH2CQuviReNluBmyKNq1C0VZIr+Y20DYMvH98GURTD3XOWuY83Kqe3up/s0xCQR5l5I7psscq+YdNTOD7V7OLhYxTDPIZXb95EiTttCaMEDTtqGAzLDaIt47r9W1s2BmvRiz/ZX/BkL3gX7i6VpPtpzLIWT3hPOOUc8ZkQYF/xIZnLNCeg0eIR//GS+YQhF1YjoGmECyD+qwvrhTgqG9HBUmnxXUmLzVZAWbjkljKYciVztCTRUnj36PCFJlfSiqpX/bAlj3cnMfD/k6OXMZzcRxOP2rXyNrqdpClQk5ZQhvgoTDo/vE3bDB6nk+/m90Me5hbRUoZdF3ZjJkzZppMMka7PmIYN4QPO1J+VhZkLJ1kWg2EL55k/B0yN1eH0PMqPBtGartc+NKTB8Tg2MmdjlqK6C/js6t3V5e3Z5W12fnb58fZTcA8pbB9p/Re8nL/6bfhwLJgfuZOt9ZqCE1jfxxAQ56HgBHilT25Oo/KaI/B4qpicq7wOJzityGZrFtNMJ0ov8G4i7R6EccwBZTEOe6Hroe1c+SDs0ilxspey/jMCxsvq1eGBteW25aQ7ddKhOmR60WNOnYUz9yH4LNoAOY/sXLnklOT0jpwQhgT/++dfqTSuLYUy+OGCCUXJCL6nE/hBT/vJfrwYU42qKtKJ4xMGH7HChsvWdal3An/mumhb3L5n7RlogxDPujUVZ7tiGr2GycOxcqhqnwZjPyVtTRX5OZ7Z/7C+7nT3eHAKZi9kQY/hPtcDR9MhM9jyxxgCNz9Z2rYZw+9EDnbiWMhSbckr1VQc+/yOE0x9oDsMu52ls3ettlf/M4u3Xh5aq8MuhoCLkSCGH097mzIy3xHMkjiNbDeVQ3WU3rmw6IrFNI+HTu5AW5JHd4EbDO4jKJWGHISEbqyy7re5j6k07TZJdovA1zLpdGFj+RPEzn7R9gTOYxnmea8xX6Wt36+th8FzMzJOpodm8cDuzAItGrIj/fB1l8rnjGKkGr5Oqab46pQZATYgZ4KxMTv9p2DmXbAMLlxluqCmb8EI7BL44WH2wlG8uOcW5sWLpyAxVouO28Y389cbQVyMD+9+En43Y/xSXakVbOlTrgPx0BSVbgLHpWc0DG5dk8MlXoXtAvVYbtRCizYJYvhtPknr9GjHLnDDPkpMX5biMWnUA+1YiSdwYuBPVkpImLz0eU6xD1yH2nUTVvR1L6sEPvYSOVb/4/pjDNeXH4fDiz/O3l7vabuXYURbQerfiyWqIxkKlbxdWTKfr8LSNTJhFEU/m3LVGe4vy8xqlKZThlzlt0vuGo9wajOXh8LDL+FiuLpxN7st67YxuCtduC50uk3fsBGFN4DuDdZ7FhAlH7e4CpwjQAR/h7/+9pygL/zerzdiFDMeSbnTVAhbIQUfs/31N+geo7W4jcVdm7gGrNKjw7uIGbwMohEFYTDnJyUHYL5xkXiCaDvnIki0lHSky8z1X1vg6rRaGF/yjo1Gty4y19W7tRjFUHCeTl3NPwnrPDs8IAKOwM4jV8fN52typfmtspenK6O0DY+cDhOdFceMdUS+4x7STbub30eb3oCLSi4fWJ3Qsbgbqfbz/MGUouoD+cRnEKu67cc/zSdWdZuMshNYuewt+J1Lyrc7ctRDtkMAf4ftN7I7qUarBXIzasVOvtu2lYjun8l/e7Q/0/7QxSqkU4uL+8hnUcFodPbfYbOtws8suJPxPOR2UrEsSbOtstakvjVtTQwvo92MjRX5nO2OS9ZOezidOYddoiwa0uFvL19vcplVKmtQVxRm+0XamJh+r3uNEjiGT86d+TVm2GJtnJf/2OpvnuDirXd5FjY9HcWum81mvhmb9HHRbCb4hNGXxpCmEGRZi0JmWTD8kITPoPxZy+7PCa6vvtwGnAzn8yEid5rpys2PXPic/Cu/ZuPCfe/3JD+Y+dOkl9O9DHkkDeaJ+xe410U25T8xFLToq/FU6P8AUEsDBBQAAAAIAB13aV0WxxfFdAAAAIkAAAAUAAAAd2ViL3JlcXVpcmVtZW50cy50eHQlyEsOwiAQANA9p5jUtVgxcQdXMWP5hDLMkAI23t6Fb/kiYS/OPvSq0uS8ycHOGqNXNQJ3OSLJed3adNbo+xMALtCxYkEGHzghwzLi6xOOnoUX8Bmq+EA3wnegrvcuDHsu+G9ImJDA5zpxKJ61fVXLRHKqH1BLAwQUAAAACAAFd2ldSrwf1cckAABodAAAFQAAAHdlYi9zdGF0aWMvaW5kZXguaHRtbNV925LbRrLge39FGrJGpE2iSfDS3bz5SLLH8tjyKCx5LuFwWAWgSJaI2ykU+mIOI+ZpP2BjIs4fnF/Y9z1/4i/ZyLoABRBstWa0Z2OlUDdQqMqqysp7JqDFJ2EaiLuMwlbE0epsgb8gIslm6bDQwQZKwtXZIqaCQLAlPKdi6RRi3b90THNCYrp0rhm9yVIuHAjSRNBELJ0bFortMqTXLKB9edMDljDBSNTPAxLR5RCBCCYiunq9Y8lrmuQUfvv7P+BLKuguZ/AHmrAcvi0iJuDP5B3ZLs5V99rcIc0DzjLB0sSa3sB4J2HsJIwbhAEdn/KYJXdk14Md5SzZ9CBJeUyiLoQ02ZAEgiIXaQzPv/8eQpJASDIidiQBTndpTJOQ5AzyHUsCwqmLu4hYsgNOo6XDAlzGltP10gmJIDMWkw09z683n9/GUW+RX2/gNo6SfPlkK0Q2Oz+/ublxb0Zuyjfn3mAwwK5PABH6LL1dPhnAAEYejLwnq0XAeBBRCG6XT4bTJxDcqd98+WQ4eQJrFkXLJ4+9kbee+pPJk/PVIiNiC+HyycvhFC6CMUxgCpcwheGQTGEKCHvYH3owCAb9EXj9KUz7w+GvFqz1xXpEKcLCda1wr7m4wyOY8TQV+zOAft/fzB6tp2uPBnPo9/OCr0lAZ4/W63W4JlZT35s9Wg+pT0fYyJLd7JFHvLE3wNu4EDScPbq4mPrTITZELKGzR3Qc+uFgLuchQUATMXukNjgvWxSotfxjtebpWswe0RFd0ytsviE8mT0il9PxkJh73WntrwdUzhqSZEP57JHvjYhHq5ayI72iU7WclEV3s0ehd0U8OW3I72aPgsH0YizBK5qaPRqtL8l0ii2p2CLoi/CCXAUKBichK/LZcJrdSkRtSZjezAYwzG7By26Bb3zSmQ5640HPG/TcwbTbG8AlPhy3PlVQ12kiZgXr5yTJ+znlbN3L73JB437Ben2SZRHtq4ae85puUgo/fuP0fkj9VKQ95wWNrqlgAYHvaUGd3lPOSNSrYM3PDmef7f30tp+zX1mymfkpDynv++nt4QxlSM9Pw7t9TPiGJbPB4Uze4pr6axIj0q4J76hVduc+CXYbnhZJqJv9TXcepFHK9T1Ldt05EkN/S9lmK2ZDdzKZ92+ov2NCAunncZqKLS6FJChgGMlpeDhj8WYfk1slfWbDweDxPGR5FpG7mR+lwe5w5hdCpIlc24wlW8qZmAcFz1M+y1KWCMoPZ+4NJ5kNZ+gNstu52R6QQqTzjIQhzj8AfHg4c3O+z9KcoVSaET9Po0LQuQaQ3c7NVrLbeXpN+TpKb2ZbFoY0mQcRy2acBqIzkBw66B7Ozs4/AxTGlMNn52fqam/m9JBQJFUczlyRZnuzyXVEb+ckYpukzwSN8xmyBeXzd0Uu2Pqur4XlLM+QOX0qbihN5huSKXrE4X3c/Ax/HM5cn5MkfC90Od6Ti5EDYDtUh5+zX6lcbIm7eUSFQN7KSIA76buDIY3LgVlJQzWCkJKiO69gDkfuRCFdEFHk5QpZIsnmvoVe4j4tQNlteZQX2S3gPuaavDWrXl1dYVuTarWI6+reeLKQpxELQT3HlXTnN1smqNwunSWpRmuYir2ijMuKMi6PJp4MHtvTPvJ9v9yyG6UEFw0S2NHiUNJ15yRhMZEEmRVRTmHoejmwZI1amVagOCXh3QlAWk9WnSnnKT/RWQnO7uHs33b0bs1JTHOQM+8ng8f7FM9c3M3c0eFwduZuKU8rip5KivYUFeEj2HoWEQURibMO9uqNr296o8vstikihpOKQwcwRI49JjaPxvOKsS806+r5gMZ6SlR4M7X1GiEqPdM1Q+4n12qe6XhQp7qpYhaSJJSXxJukCa1t4dKiTaRLkExapxFFr+0HITVYXbTqE7LXMi5hWiT8aD0IrgKvXKSbb9ObfUOW6mcQpCG1iUEqqUEP/7qDSbfaQ3YLk2MqR1yguIvIXVoIFHfuhrNK7ODNHH/0BY2ziAjaD9KoiJN8NlxzGK4VY0sZbTN9LggXhzM3IDw8ptWHsW99qeqJusFnt0Z9a5DyptqvpmacH7ajvX204yN6mD9IympVI2jWlHkSS1mEcr02TFEgGg9G0Hjj90maGrG3MIDS0Nb6PRTHquFGTXIxGKhTFcTP5ZniRaugPnU0fa+Jf0nsBrvjUq30/VSINJakfJAT7fWhDmzggpMkzwiniSiBXLZzFQr8UwpIb3A6GDS56CEneGGW+BPhjPRzGtFA0HDpCF5Q5+d76PTIQrLoT5mPI2MgGt671IZEyNPs1zSh8hzwxuAHhUpI8i0N4VE4CS79qybGx/eoPjwgkmc0EH2OamY2Ph/Na1zbQo+C3oq+RI5pKdkFGbhui83lmSmbyh1O8nlpYXEaEcGuadOYOqgNzrbY3pPXLl7qHfdbpPkpylfi0wBcp0GR969ZzvyI7tNCSG9l1JAaBqR+3k/X65yKmRIECAdc9Be19p94FVdOvKaJqXTYB5xH973Ib9dlcl254GmyqYt4m8An2uJSnTOS7B9moWWcomtb13LHx9iyzaaZbJtCg8HgiPSqyQD9AMsHMDYCXqf+Oxy1ZmKG1jBhiTXODbYsazHjI7oWM30cStQM6udgs93U8Co6qDUpWZdecHXC1LQMFFxwAw2SgcpGGkUsy1nebmiW+7pt2RTXSMlu5yLNZha11eWm3Js3mfTMP/fKaxVHxzsxu50ay7pheTcEqlSW8UcmleFw2EYqAYnhmoU0fRChXBtZtE55PJPRrL90+sOuBuRuChbSFhyzBNl/8F7G1PKuT69pInK58xpoCNm1Xuh4+ri+n9H5eH4sz49P7aJ5RpPBYzNJvsXYzgnCl9ZBpWItRMgrtMj+0ulPBo+72tqYWnJteuxQKXhywWPL4lyv76c6b9KtLVfJ+CPDszZmXI2J880DDkjR3CnJ+YhSep8Ga5h10h45c9NM5B/DRbc0xFD5S8qObHrt+Q0TwfZhbvvgaMVNFVzk6ElIO8XQpZoAWJIVYk+yjBJOEil3EhPykB6PMThH7/epH4WXwToYt7D7eyyCKuhUW9ZsRtaC8r1Bp+PMjw8fxd5I+olrIS+0KLiw4jUX73XLkXCbqzqyzbyGbeZ1mwsOtjTY0RZPpVLUbf3NRuUeTChK9ctjEkVtavqkktdBHJFYBvRpCxxFuPJSG5Z/zW5oM1je51q0xO8eyjcWcV8ZWVUdjNqdkhyzJBWdWchy4kc07O7XLBKUz3ypFxOa552hMqHlENPPPp9HwTqY+r4h0SRFuRClNxiKdAXL8n2dXQcV6sal4/Fh3k9DgR4frp4Z/H1TQ5snRWSW1VhVX5k5l8Yh5zQvIuWQ0zgTd/tjyddCXKWFM8AtKoKUw8H1mTHLLiwNcdFi+T7QRbVw9TDT95F/RcjV5eHMxM/qxsZp0T4xu2lHeZ4xGcvRws/ytscnttLUf6cOX6RZq9NSxfZwbnAvc0CPg/AywNeC01pkDsftRbqv1DlPBRG0M5oOQrrpHg5nriKBGpJUax9D0g9TMSpMjWm3/bF0N5xuYUzdIMhyPgx05teb46X2r9RSdR/3mkT/ghV2fPy2YLusCTYZ1q5FIO1FaNl7UtAOjgImGE9oDSLKCEelqWoRTXcwltFzcyDg0jvq8/Rm33A75M4q7BVZRnlAcnoM8JLG7wt9yNhONed2bOSJSkvUjSCM2Z6M+guyaYaE2nTSkX4xXDm6x4uqfDDM6BCetAVaJWKltCkls2T0Y4ul6fFVcfaWMKsKv7d4wz7hRinIqWXMox7otJIphPe5FXa9LxDqyWWvOUym2YdYe9U04GJ2fX/skvmE7w2Nt0abT2osdRDHERofpVSDOWzXSwmFpu2hoFkKXXYD9yKHoPBZ0Pfpr4zyjuv13Mue6/Wke1buLguErcCkllfIuMZQXCL6SRFTzoKZIH4REY73ebu4x2KEuvCrG+FlpsmmMiUHZR2DjJ8cxU/vQ+w9inZiNFMLulD3cu6ntw2n+tgM+fhZBWNEBGksSyjwzGQoOKcBXtf4QCJHP4AVhsstEVLLvioBc0KiVBAyNy/8Wsx9ek9QVy9acymnwbHKC/oflJVwPSs3oRM+BghsJ/tmrqq5jAdFky/rYCvLrhKRg3nE8iqlldBjcXPZJhVKoBGrU7qxrO0lSeo3+RZrpFTapSZvKDO7q5VJayOiLctFyu+kKaqvmweEzQ85HE4zSkQHTaI+Fr30YpbE5LYzHA2y295wzbtdWwYjWNBFA/dnit6TYGr1oAbt0T0lodAab/FfrBWZCMjp0HZLqOavnf4wu+0aSI1QaS2yNDwKgplRGI6q5VAaJOxpTjKdwT9tC43MKRPf5P8w15x/wGmO5cE1oyRlXmha1gp8nDxg3WAwMlRP0Wr63S93miYO2oIPt9gm0pSSU5/GsiVDlcC9xypDevznka8BgMq7lliaWk+SpnnaNLPb8xRycHOL95iepTyXA9tFzCkn+mydphhW0WNGTQdel+CMLL0rlXwr9bwnU9IU9v9ktc7Zv8U0ZAQ6Vf7gajDIbrtYsydT6r1KjZ3UXAfsLPmvpwjh3sz74ezQMu3EK6eVpVRVrZQiA2gSx6VuNRZv+5xXU2PjDnR/2wepu2bqMfqcutDGrrQx0z1om7jFxbmugVyc6wpZrHDT9bJYARGRPF86uFlndQawQJGnG0Wq2uqtst5JtwPIIlFVP+uMBw6ohaprUxjqlIWhDsjcsdIYOnNs14o6w6mDtaLyN186w4mj6jsdXUjpVJWizgdWipaQVPmnY+pEzUZCdr1abIdVfe/ifDtcLbLV6fLc//2/4HlZfrs4z1aLc4SiUGZdIvZYuHRUIZJjMKluQcdzHOBpRMtOqwWa2qZrmAoH15uRRD+o4L2ht8JZvaRxQQTEaUij3/7+n6avWYW+UFRA+ersbBETltx7/lgpZAhg662+pQmJamiIixksaLxqqVImghS6VHlxTuNVD4JiV2QQEs4gJ6KAdSpSd3G+9fQM2erHZLMhWzWUxD6LZB+Faxde4s5knfMdSTawIzEDDMVsFcwkjYBg4XNMkw0LaSLYmu1I49xC2SH2KWfNImleCAw+kbwsl1YT5TQvCHPxeCs8nlmnqmqJylM1t+o0SUS5PLraMN0VOdcg+JN+H775/tWPb6DfP2Y6FDoVz21HderAqhpnNdRnDi9JXuxwexX6FufbkU3pJYsTPzdLFcRHg7ucBmChjDWrs9VX83KjDgSwkLyPVLh0igxJ24JXFxfI40Zc4HVNXHhj8MaGZdFQdmSef0eXTlBwrIR5jqrJtPY1TK9sQP0VkGzpSJOp1vwuZYlpt8SJN4Th5HpMPPC0EPHAezGx7/vedX9cl0EXcBn1J/gXGtLJg9H10CvlzI8SG4tzhdN/FslrEuV1LAckppz8/4PlEQyvGjge1XH8p8vqHry+tx1HXn+0nUYejLZj+xl4vyJ+bQXiaQUykgpkXKL/W4mmJvorvqwJaiwVKRla3QjisySkiDhzPgqUPqKI+PhOxSsWsa3FeDY3lQrhmA/lqxgr+9w869y8/3fnhsXdcLt0Rg7cyZ+GrC4tsrp0gMs+9bO4UkdxJU/CqzGH5DXJN99NwBuWp2RpTUPKsrpn9ZpyKhRedxRyljClJ3YR20GRiGKHch2RjxaPHFKDgoLxD6++7sGr77/W2unPXz17hSo8JrvcheEAXj7DW5/lBN4VGwJoU9EIOs8Fjz7/U1eL1+o87bXWb2SeU1LSmkXUAXxzyVyjV5CJpaNetnmX0U1PXWaJubqhfqYv/bgixJw7x6SqC2XKTubeWieLN3bXb+KNAyQSS+cVOsvJO6KUsT2kNDH0mO9JTMspsNSotEaO5BiOCiJKKo14axCgGWb1Nb76IOd8ID8GJK5mJ7G9UlkJIzvJK0emjNAjAfyRq9g8SN9ltTiXfeosWSpjrFZxVsoOPK9+tm5QlnFUB6PuapusSYWnpTWDeDvSANY2X+abEmycb5xVG5W1KHKs1LDRIicu1yez6s7KokshiFmwTMf76a0DOjF/zDlvaC76b1hM4WmxiWkiZEh0IeMFqx9QF3EiCGSchgxNZcmnJI8YfA7XlOcMAmkiLs7VkNI6lcts5yEL2yQh0d2vFQH6Imli2yS5bRq2lGBDWn0UYer+E+LU1lRDramGUj5eNOWjN4z6Y3eE/5q+is2gGjfKCXiKFnpuXkQssWwIrs0AZBn6Gv7qDSbY1bltCIctyRka1AUnYnHuWzMX0WoRsZV8yRF8PNwtCJJkBAUpLTIlXFVJwuI8YrL3c7IldwRIhFa7NKtRGwvSgy1LpP2O4a0IT6gco2aIaUyTYssgpz7ZMJKAT3PC5VJ70tjfkpBk6A9I/Y6Dz4s2mrKZR5rbP3z1+sfv7rG3NQuza7p0sjRigr7PBPeMCf5Cos92Q44scDw8WV1QkrW6a5dOPtvU7YPRuCJpvP4YJD10L/81kjbGlydJ+qrdHhgO5FN3ahZYW1LDnJs8cEzJOZfuBIaTYIjmoevB0J3ACJvcSe65k7667Q/dySmbY5GtntGoiIGEBIhmKHfh81WLXwURiQoQFFsX/hH/+SvjNx7LN0MDpfuvT9rcr+zj11UaRh1UvnBOQ8VN+DKwnhyXJ0MAbdOppFpFc/q2zVttjFQVFeVIfdtOria8VXdL7A5yi9bDhsAeDB5bIlve1Sh86Ml/dfqbDhS1TBW1TLx2FtAv9h5RP5qotRUBGOhy/yzZPOWB80FT6bDV8VTv4TOshSWcEzS6val7UXugavOtJ3fSXawSLkd9ZZXNqVzzsXppntY1iYxlVCqeIE3Wf8L2weMyMCWV+1+ffvvN95Wmb7GkGqxgZjs9v64NcVZ/sGI5gvJQheaOoKGAHhuifYPv4Tur3/7+j8X5dtykObMd7EgqLhRko4fUrdyjpTdvDbdgIUUJTd6s3rBksyMCdvSO7FiCoSjJxxwjUFsXnqc+UbKlwC8blG/4K90ZUZ9ttdqUcSwtgGRzSBVgrQbdU6vCKg4rUsVz50j8ma6Y7q9MUbxptUVrl1aYy+SzFW4DS2wEZUxxtPqh5SsF2p2zjyawz1AbNUadLrJSSBa+s3pFkrAgCRRxEaN5EpKccBTP2yOVbEcGKzldF2Mq6eDUhGEtJrfYTj6uzXlCZPyLxmaLZrYjVZdbdzB8ORzC0NsOryfbyiN/zjjrB4yjATNZLTJzItiOFJG1aIv/ywhSXyn4KPZ4ZXAPYPrdFQwvZFCi3P0PZVBYGq3oZXCS7EiisFFEBh2yo6NMz/92hKgPQHx0B+U+mpm4U5i402jouZeAP0qUvaY+YbvkjkDItHHfRNYL1XyMLkuQaPlxLEx0FUUpUMy9JVTYDbkjojTeTsqKN+i9xChmciq9Wo6Lxog/w++/uPCtHV2K2FbK2NjHNIgUJ8kdaQgOs8La8pzVezZWrf0NLgMNOtQMJxf+PE2usUCUpQmJ8NMXXP4SNynfGUrFFxvkFX4VhtUzJT3tsaksPeWA8ZqEJZt2MSizja0yEJ84RvE/5TkTdCcKXmp/f/UDzVlYkEilyPzTzFED9Ypgpa/0IA0kFXeRD3KtCR4K7Sm6sTkDQXPRBPg0CD4Q2rc0InkTzHPsSFsX1swEapiYu71Pr9jeZeKsBia9s/BXrzjJaMzTnOYoiXzMUv4e7YaQFT7Zoh3ww9fP9Mn3OcXcMroI70jILL2av2a/UmfleeP/+g/PGxvfNWO7nEY9yElMIEMWySGnGeUCM3dEINWwkljeJ+ya2/CqbXy1ywUnMl7EJNHIjYzBj9Id7BSJY2LOUFBn5P32P/6nN5nq6EIXgwMZjcg7JGu0BEXBe7BjEcl6cvdZypml3z90raNqrd9GJC89ebnOr6PUJxE8vaacbCi8StNIZj6/xNxxTzmGr9O1iMmtDlPkLEJDJOOpT3wWSbUiGMmO7JAPXee4WqdlUOl1SnI1tmR9bjzSZLNhELKMCpk93dGHJELfI7IX55hbxhyzqkU5zjLL5b7i6R3dwdPdlnF4xclOsF0Rw7c0UGZbAs8KIkiCUfkXNOF38DLd3UHHGw+uhoPp4GLQxUffJFhnRATbEfgxYTLeiKt/WUTkhvAYWcQY8WpiFZ2Rsas1EUDDYkcEW0uS8YudZBuySVKkfCwRyV0DYXGudoR7U5/yWp11Ol1YrgCLRoI0yQV8CkvosFC2hmlQoGpxN1R8FUkt8+zumxAfz8sBz//43R9/eA1L2IP8UBM4quIGb7pOD/BrTaYt5LJJf7HJtOqPnThwqKC++OovNkj99acSnP78kw2rtKwsKLIC6I9r3FIkd6QW+1P0M/ztb+VC8WtRXacataW39pgXX/3FDNCflLL6vnz6l19ePoMlDAdzVIoRFaDDO7CEpIiiueKE83PY+1Hq9+TX1HpQ8Oig+6sPwSxBpkfnutEv8qO2XHBKYgO3Wq8u01zCTz/LRZx/Bv3yD+hKDeWuWe2fnZ8BrItEWSc5Fa9lx86OJWEPsBCuK+kC4NOOKezoupIbMKMCSzBFIA58DjhqXu8tw7pdFyE9VzVVsJRwsR9unuR3SVAtAYNHyA8dIjB7hb0HZgmC3+krs2sOSyA3hAlYUxFsO845ydg5S9ap04M9BCTY0hk4SdpH7FAHDt15DUBYAuDuuzxNOuVztoZOqL7P0y1nhfKcsFZgXrZWiHNkB6cHjgpx5YxkTrfq+WnH1FdoNH7HcuFyGqfXtIN5mJtGb2MwNHEYupl8AF+Ul65Iv0vx/ejXAkM9HYeF/W++dLowA+e3v//DqeCqzZMgkIDQqviFBEHBSXD3ixAEvvii2VxfE9oczQUhtE8UWcIX0MHbz2A4GHRdkf6e3dKwM+zC5+A8do7XgzCNAXK80UA9cWOSdTqB5MfAlYmXrovGf8dBKdrEm7QNmsDefroPXRZvfkGL4vBf/1G7fWsAHIDiN5v2becrP8JUna8vI62C8hwr8ew1KBz7sLTPvHrsuzIg+uLNy++Qixb+ygYYMiyGclEDwmsSsQQW+KmfFSq0XzBx9EuQJO6OcjTk5BMp+1UniZpc0nP1kGNSK8KKOdXp1dNf0kJkhcjPpVQ4N113tN6x9lSbBe9IJJ1IHefJKb+mXIaYF2V+TP527A3/e0H53WtZCZLyjiN7tBy3+swVitra6IpbSBg2WeVwps4tIPj2bYdWLNt2dK/lekGwkOzw7LaFXyQbp8b7Rv4sYNJFIJg8TAuhdeWRmPocht0e4DclNRS1oiILiaDPZPpKyZbDsXBWSU0ZNzspmuuAKplsMotd1yQQYQmfdJSY+t3vSj30u9/BJ6hOuvPGyFYBLRXPF+C8tKLyv/39PyXn1lMEjpHjlRLZpjevSEI7N1sWbK2lyjKUriuDva6uuoUlyG6wXC5BF6p8AaquDKTkTqhTLtmUBtwLpKwn+AIcWafcAghT8PcCkfn6dgCHhsb8BsscOpVaNztGIvoEm13kV1e+opH/mYltR1dOON0ucCoKrnD2FVJnx/k9i6jxgDNZi6MMug2JfcJdQ6QIXQKXrtHK2CAocb2x/tUC/u2P6Ekm6H9QpHzJzSof2VFlJJ/uFagDvHzWdd/q6SpzZg/VVmfyp2RVWQz5Lts40qqZwY8/fOcGnBJB/yjfY/jxh+/kervKNLOPE8s5um7OURnpedyCR81usoKjSaimP65DDchFmj2XMWSjy0uCrOinhK2zR8e0YBNMGxNbNCBrRRQVmLNvWH9nlT5AaYpSzljVdZn45PTnqZ50XayZy6lwEUZFBQqiJlospOuCpLUSCUqZVVhQfHh6a2c2t5Iw/Ao/jIKClyaUd5wgYsHO6YEShJ92VFUQWjMs2HW6Esh943f0LkxvEoRAld8ht0HdHb1T+/gKX7JykKysRnC6sAcqv6tDE/ElXZMiEri9oyXM4aANvepRyz62+OqcvQx1Qmj1U1cQvqHCxdH5T4Of53KN627F8OserCXZIYJN92sSFdIoduZ6BT85IXrXckc9kDf4Mo/zs7tO+Vck2HY69Npg8hTO6LW1yhMo0GMbWlLOhQjpWquJKLmmajVp9t+wEmPdNhZzH5HIByeOBtngjY77PfCA1FmUTHc8X0ZyYdOCxbH4mh26Va7rdiiSWOanhIdfEkG+cOU7eEipP/3c/dldsyTsdJgEwe4R+xXz4nhrzXiLjvbTHNVAp9sDR9XwkcTNjIVSkrYqUruPRyvR1G1xClWq7cjmaJdNT6Oo46B4croVwQi51TaE1qWEwucHAL6Vw27dnIqnQnDmF4J2nJpgdHpwK0WDsBGqJa9RfXPpdEvTKi9CsjVKdQaCCpKBIHFmglq6lLAEJWoC9wEidl9XPsciV+oNTf0Nv7cGT+OrGi/tlZoayTfKsdAVd/phnG/u1WSfdnSpX4vCU6ZOhclPEnLNNkSk3JVvGX0pvy6ff4H0+WNO+UtsrOxsnLqum51nPL3JKX4Vg2lbGwN7xa5INkB2Oc1Nshe+LhIZvBPEB1Vm7jrz9t1I03B+707Uhg0FWNa4HUAo4yjK/W/fbG2vnb36atkM9rCW7+2hxzYDBz/T5PRUEgwfspBiNGroXQ7g0NMZMevB1XSAGqoHpAhZOlPxHSs08WlHl4JKq0jZT7DUK56f9HRaTkDVjGvky7rgkEnMu/DNr0x6cPZBYO7I14fW0U1bgpkwn/J1kWxyjLRChEGGbYoRBKyUe/HmzavXXbd01v61YyuPq25nW4xl2dcKJV3JeXiF5/WGk2CXd1qkFAJBA6URQytnsxb3MIGmePFacaI+MouBrl3Z9meki26NHHVk0jYElbGsI6zI18k1yUumd9UHF5Zgw5xD4Criqh68kPdm1IYqargVHccL0Unk5EapmuseDHowKCcQ6bMo9TsdX+7PR4excm564GhWRRMf1VJV/O30YCA/S6hV05GaKcu6TscdS/+kEXFUBYWnWfzTTlltdj9BKYRTdVLG6JeG23FgEuh9wlETSuV3nyYVJeJtgpFkYZyDv/1N+tl1ytAhXxlebLHOH4oVXdt2b58H+T73Y9jWGdpEw/hDQm/g9ymP0UYy616HLn6uLgm1HYTWiXbelENpu3LHY7Dcu4erwYuuqwu90UUfSvd8YFjloUFiWeodCBknjqnYpuEMnFd/fP3G6QG+YTrDrVQyGUPv4VxBb4salxJZPd1DutOCvQcy7jSDtzrspOJbnU/3XH/Y/dB138JhrpWUphLupjskkU9CN911QWzxSyqIWMUoVpjsXriNeDdNhEwS7MF1XYzwb4vYn9lut/Hs7eMwPjuYNINbJPmWrUVHwusq49s8i2iyEVtYwWW37J+h5DVAsFwMfQk9Vt2+UD1Nr9ZQXikmqBvTPMesJcYe3lAuE8M7mpOIbLEaArO8MmcpI3vMRCidEvqaJSSKKlKppVoeJFdaufNQCcFSuunthh8m1x7Knac5/Zg5gzSCpUptdUIdQbcBqSK144jsuyIsolpHsmnpJgEed1MLq77+gGGbtOpnyjCb8F4SsXXlgE7oYiesfAuoTiqoTIK9OSIDSDixLnPtzrFNT6/qheypm8++rGpNlyALU1U/Tv+9oLl4aj769nss7NBh4Pue7R8wA3wGnSH0wd5g5SBL9MjKy+PTDd0ovfnFQktbsNLCDpZKmlQER7tCttRzEBqdoVsl2RnNKzsqszRZKVzTm3usmJBd41Q8vannDPXHBkqTETvYS3lSrxDAyF/5yo39TgHBgnJ8X4KZOl17XBZU75w/sadqJCKkjDuiv0yRvcup/Fxdx1H/NxbGTapMQRs0nLYJrJNJpJ7KiNWFNJbIwbINNkNs4uMTTIWJ7k5W42xQB6306PMti8IOx/8poZTF/yx1W8swxun7dlkR9qHuzeKHuSRxyjLd7ly2tBLnT2//THaiwC8tUk6TnIFM4am7gP4S5weI87c9ePvmzVP5TOYzwcHijLVhDnVzwG6y7EilAcmGykTgT4Ofy9Rg2Tb8+QDZ7dufNd6a3oUJUeX3cAMSIm4ub1qder/2GeU6amVhiVOVrq1/1suSt0G7/OY00KSM+eEbyp+TnHZswa9qYdvGYd2stsVYFKGJ26mKRXuyi6y26SHLWoWgE6sQdPIRKmOnH78y9km3ZWOmsFNtTVd/fqzNnapyPbk5u8T5EqbfTWF4+XIKWLQK1ov5T2pneb8pUPOqy62zsCfjnHkP8EXubs2/LSLJm1jwA0XUypVy7CmWiNg9PBFJkRaxGlhcA4qMha6Ul78c2a2ZQ5YsdcxPRVTjpoiVQeDDsWlWmp61bW+VRJLlr905bFs3buzbKoTeoiP9e/avX/9EfSwDxvK0VFspvxuqEd+Glm8/mzdasFjTX9WQpbThkwpEU5FYOTfqSi/gZF//CMHUtghPjFA0KHVSaXPSpmZ60HG+/Su+bIKJycoipMcW4eHx2wrue+M3pRPSncMNS8L0xs0DnkbRm7SzB/yeFIZHfLol1wydN0f9t3eygshyC7c1MvMb2k2TDxZ3HzNlw106lWzGsi9TbNCdnx26+HNxbmr3Fuf6c0jn6v8V/T9QSwECFAMUAAAACAAdd2ldcSNcMdwAAAAbAQAADgAAAAAAAAAAAAAApIEAAAAAd2ViL0RvY2tlcmZpbGVQSwECFAMUAAAACAAdd2ldeIV8mDsEAABvBwAADQAAAAAAAAAAAAAApIEIAQAAd2ViL1JFQURNRS5tZFBLAQIUAxQAAAAIAMF2aV2oWKwcwA0AAEwjAAAKAAAAAAAAAAAAAACkgW4FAAB3ZWIvYXBwLnB5UEsBAhQDFAAAAAgAHXdpXRbHF8V0AAAAiQAAABQAAAAAAAAAAAAAAKSBVhMAAHdlYi9yZXF1aXJlbWVudHMudHh0UEsBAhQDFAAAAAgABXdpXUq8H9XHJAAAaHQAABUAAAAAAAAAAAAAAKSB/BMAAHdlYi9zdGF0aWMvaW5kZXguaHRtbFBLBQYAAAAABQAFADQBAAD2OAAAAAA="
HF_SPACE_NAME = "skinsense"

WEB_DIR = Path("web")
if WEB_DIR.exists(): shutil.rmtree(WEB_DIR)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(WEB_ZIP_B64))) as zf:
    zf.extractall(".")
(WEB_DIR / "model").mkdir(exist_ok=True)
for f in ("skin_type_cnn.keras", "labels.json"):
    shutil.copy(MODEL_DIR / f, WEB_DIR / "model" / f)
(WEB_DIR / "requirements.txt").write_text(
    f"flask>=3.0\ngunicorn>=22.0\ntensorflow=={tf.__version__}\nkeras=={keras.__version__}\nnumpy\npillow\n")
readme = (WEB_DIR / "README.md").read_text()
(WEB_DIR / "README.md").write_text(
    "---\ntitle: SkinSense\nemoji: 🧴\ncolorFrom: green\ncolorTo: yellow\nsdk: docker\napp_port: 7860\npinned: false\n"
    "short_description: Deteksi jenis kulit wajah dengan custom CNN\n---\n\n" + readme)
shutil.make_archive(str(OUT_DIR / "SkinSense_web_dengan_model"), "zip", ".", "web")
print("Folder web siap:", sorted(str(p) for p in WEB_DIR.rglob("*") if p.is_file()))

token = getpass.getpass("Tempel Hugging Face WRITE token (Enter untuk lewati): ").strip()
if token:
    try:
        from huggingface_hub import HfApi
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
        from huggingface_hub import HfApi
    api = HfApi(token=token)
    user = api.whoami()["name"]
    repo_id = f"{user}/{HF_SPACE_NAME}"
    api.create_repo(repo_id, repo_type="space", space_sdk="docker", exist_ok=True)
    api.upload_folder(folder_path=str(WEB_DIR), repo_id=repo_id, repo_type="space",
                      commit_message="Deploy SkinSense (custom CNN jenis kulit)")
    SPACE_URL = f"https://huggingface.co/spaces/{repo_id}"
    REPORT["space_url"] = SPACE_URL
    print("✓ Deploy berhasil. Link demo publik:", SPACE_URL)
    print("  (build pertama butuh ±5-10 menit; pantau tab 'Logs' di halaman Space)")
else:
    print("Deployment dilewati. Folder web/ + model juga tersedia di PA_outputs/SkinSense_web_dengan_model.zip")

## 10. Unduh semua hasil
Semua output (laporan PDF, grafik, model, log training) dipaketkan menjadi `PA_outputs.zip` dan diunduh otomatis.

In [ ]:
# Paketkan semua hasil (+ notebook bisa diunduh dari File > Download > .ipynb)
zip_path = shutil.make_archive("PA_outputs", "zip", OUT_DIR)
print("ZIP:", zip_path)
if IN_COLAB:
    from google.colab import files
    files.download(str(PDF_PATH)); files.download(zip_path)